# recheck_comlumn_preapare — ตรวจ layer + แตก process parameter

recheck ผลของ `column_prepare.ipynb` อ่านจาก `processed/prepared/` (ไม่แก้ต้นทาง)

4 เรื่อง
1. layer หลัก vs layer เคลือบผิว + จำนวนชั้น
2. คอลัมน์ใหม่ระดับ device: เคลือบ / ไม่เคลือบ (layer 3,5,7 = ชั้นเคลือบ)
3. layer 2,4 ปิดด้วย Annealing จริงไหม
4. แตก `process.parameter` + `unit` (เช่น `155, 600, 600` / `W, mtorr, sec`) เป็นคอลัมน์แยก

## Setup

In [1]:
import re
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 100)

ROOT = Path.cwd() if (Path.cwd() / "processed" / "prepared").exists() else Path.cwd().parent
SRC = ROOT / "processed" / "prepared"
tp = {p.stem: pd.read_csv(p) for p in sorted(SRC.glob("*.csv"))}
lay, pr = tp["layers"], tp["process"]
print("โหลดจาก:", SRC)
display(pd.DataFrame({n: d.shape for n, d in tp.items()}, index=["rows", "cols"]).T)

โหลดจาก: /Users/farlos/University/Perovskites-Research/processed/prepared


,rows,cols
devices,44,4
layers,352,10
materials,543,13
process,702,9
properties,563,16
stability,62,20
vendors,644,8


## 1. layer หลัก vs layer เคลือบผิว

ตำแหน่ง layer_id เป็นระบบ: **เลขคู่ = ชั้นหลัก, เลขคี่กลาง = ชั้นเคลือบ** (แทรกระหว่างชั้นหลัก)

**ชั้นเคลือบ N เคลือบ "ผิวบนของชั้นหลักก่อนหน้า (N−1)" เสมอ** — ตามลำดับการ deposit จริง (สร้างจากล่างขึ้นบน) และไปทำงานที่**รอยต่อ**ระหว่างชั้น N−1 กับ N+1 ยืนยันจากวัสดุจริงใน dataset:

| layer_id | บทบาท | ชนิด | เคลือบผิวของ | วัสดุจริงที่เจอ + หน้าที่ |
|---|---|---|---|---|
| 1 | Anode | หลัก | — | ITO/FTO |
| 2 | HTL | หลัก | — | NiOx |
| **3** | **HTL treatment** | **เคลือบ** | **L2 (HTL)** | SAM (Me-2PACz/4PACz), Al2O3, ionic liquid — ปรับผิว NiOx ก่อนวาง PSK (buried interface: ช่วย wetting/การโตผลึกของชั้นบน) |
| 4 | PSK | หลัก | — | perovskite ABX3 |
| **5** | **PSK treatment** | **เคลือบ** | **L4 (PSK)** | PEAI 15/22 dev — passivation ผิวบน perovskite เกิดชั้น 2D อุด defect ก่อนวาง ETL |
| 6 | ETL | หลัก | — | PCBM/C60 |
| **7** | **ETL treatment** | **เคลือบ** | **L6 (ETL)** | BCP 37/44 dev หนา ~9 nm — buffer กัน metal ทะลุ + เปลี่ยน Schottky เป็น ohmic contact ที่รอยต่อกับขั้ว |
| 8 | Cathode | หลัก | — | Ag/Au/Cu |

หมายเหตุ: จะพูดว่า "เคลือบชั้นล่าง" (ตาม deposit) หรือ "interface layer ของรอยต่อบน" (ตามหน้าที่) ก็ถูกทั้งคู่ — คนละมุมของสิ่งเดียวกัน

In [2]:
MAIN_IDS    = {1, 2, 4, 6, 8}
COATING_IDS = {3, 5, 7}

_L = lay.dropna(subset=["material"]).copy()
_L["layer_kind"] = _L["layer_id"].map(
    lambda i: "main" if i in MAIN_IDS else "coating" if i in COATING_IDS else "other")

print("layer_id x layer_type (นับ device) + ชนิดที่จัดให้:")
_chk = (lay.groupby(["layer_id", "layer_type"]).device_id.nunique().rename("n_dev").reset_index())
_chk["kind"] = _chk["layer_id"].map(lambda i: "main" if i in MAIN_IDS else "coating" if i in COATING_IDS else "other")
display(_chk)

print("จำนวนชั้น (นับเฉพาะ material ไม่ว่าง) ต่อ device:")
_cnt = _L.groupby("device_id").agg(
    n_total=("layer_id", "nunique"),
    n_main=("layer_kind", lambda s: (s == "main").sum()),
    n_coating=("layer_kind", lambda s: (s == "coating").sum()))
display(_cnt["n_coating"].value_counts().sort_index().rename("n_devices").to_frame())

layer_id x layer_type (นับ device) + ชนิดที่จัดให้:


,layer_id,layer_type,n_dev,kind
0,1,Anode,44,main
1,2,HTL,44,main
2,3,HTL treatment,43,coating
3,4,PSK,44,main
4,5,ETL,2,coating
5,5,PSK treatment,40,coating
6,6,ETL,42,main
7,6,ETL treatment,2,main
8,7,Cathode,2,coating
9,7,ETL treatment,40,coating


จำนวนชั้น (นับเฉพาะ material ไม่ว่าง) ต่อ device:


,n_devices
n_coating,
1,9
2,19
3,14
4,2


## 2. คอลัมน์ใหม่: เคลือบ / ไม่เคลือบ (ระดับ device)

ธงว่าแต่ละ device มีชั้นเคลือบตรงตำแหน่งไหนบ้าง + สรุปรวม

**§2.1 แยก "เคลือบผิวจริง" vs "additive ผสมในน้ำหมึก"** — ช่อง L3/L5 ใน `layers` ถูกใช้บันทึก 2 อย่างที่ต่างกันทางกายภาพ:
- **coating** — เคลือบทับผิวชั้นล่าง มีขั้นตอน deposit ของตัวเอง (`process` ที่ `layer_id` เดียวกัน)
- **additive** — สารถูกผสมลงในน้ำหมึกของชั้นหลักแล้ว spin ครั้งเดียว ไม่มี process ที่ L3/L5 แต่ไปโผล่เป็นขั้น `Mix` ในชั้นหลักแทน

เกณฑ์แยกแบบ data-driven: มี material ที่ L3/L5 **แต่ไม่มี process** = additive (ชั้นเคลือบจริงต้องมีขั้นตอน deposit เสมอ)

In [3]:
dev = tp["devices"][["device_id"]].copy()

# ---- 2.1 ชนิดการดัดแปลงที่ L3/L5: เคลือบทับ vs ผสมในน้ำหมึก ----
# ชั้นเคลือบจริงต้องมีขั้นตอน deposit ของตัวเอง (process ที่ layer_id เดียวกัน)
# มี material ที่ L3/L5 แต่ไม่มี process เลย = สารถูกผสมลงชั้นหลักแทน ไม่ได้เคลือบทับ
_Pn = pr[pr["process_type"].notna()]        # กัน placeholder แถวเปล่า (28 แถว) นับเกิน
MOD_KIND = {3: "htl_mod_kind", 5: "psk_mod_kind"}
for lid, col in MOD_KIND.items():
    has_mat  = dev["device_id"].isin(_L.loc[_L["layer_id"] == lid, "device_id"])
    has_proc = dev["device_id"].isin(_Pn.loc[_Pn["layer_id"] == lid, "device_id"])
    dev[col] = np.where(~has_mat, None, np.where(has_proc, "coating", "additive"))

print("ชนิดการดัดแปลง (additive = ผสมในน้ำหมึกชั้นหลัก ไม่ใช่เคลือบทับ):")
display(pd.DataFrame({c: dev[c].value_counts(dropna=False) for c in MOD_KIND.values()}))
_add = dev[dev[list(MOD_KIND.values())].eq("additive").any(axis=1)]["device_id"]
if len(_add):
    print("device ที่เป็น additive:", list(_add))
    print("หลักฐาน -- ขั้นตอน Mix ในชั้นหลักที่ระบุชื่อสาร:")
    display(_Pn[_Pn["device_id"].isin(_add) & _Pn["process_type"].str.contains("Mix", case=False, na=False)]
            [["device_id", "layer_id", "step_order", "process_type", "description"]])

# ---- ธงเคลือบ: นับเฉพาะ "เคลือบทับจริง" (additive ไม่นับ -- ไม่ได้เพิ่มชั้น) ----
COAT_MAP = {3: "coat_htl", 5: "coat_psk", 7: "coat_etl"}
for lid, col in COAT_MAP.items():
    if lid in MOD_KIND:                      # L3/L5 ใช้ผลจาก mod_kind
        dev[col] = dev[MOD_KIND[lid]].eq("coating")
    else:                                    # L7 ไม่มีเคส additive
        dev[col] = dev["device_id"].isin(_L.loc[_L["layer_id"] == lid, "device_id"])

dev["is_coated"] = dev[list(COAT_MAP.values())].any(axis=1)
dev["n_coating_layers"] = dev[list(COAT_MAP.values())].sum(axis=1)
assert dev["coat_htl"].eq(dev["htl_mod_kind"].eq("coating")).all(), "coat_htl ต้องตรงกับ htl_mod_kind"

print()
print("device เคลือบ / ไม่เคลือบ:")
display(dev["is_coated"].value_counts().rename("n_devices").to_frame())
print("แยกตามตำแหน่ง (coat_htl/psk = เคลือบทับจริง ไม่รวม additive):")
display(dev[list(COAT_MAP.values())].sum().rename("n_devices").to_frame())
display(dev.head())

ชนิดการดัดแปลง (additive = ผสมในน้ำหมึกชั้นหลัก ไม่ใช่เคลือบทับ):


,htl_mod_kind,psk_mod_kind
additive,3,NaN
coating,26,22.0
NaN,15,22.0


device ที่เป็น additive: ['10.1002admi.202500231_Sample2', '10.1002admi.202500231_Sample3', '10.1002admi.202500231_Sample4']
หลักฐาน -- ขั้นตอน Mix ในชั้นหลักที่ระบุชื่อสาร:


,device_id,layer_id,step_order,process_type,description
20,10.1002admi.202500231_Sample2,2,1.0,Mix,EMIMI
39,10.1002admi.202500231_Sample3,2,1.0,Mix,EMIMI
58,10.1002admi.202500231_Sample4,2,1.0,Mix,EMIMI



device เคลือบ / ไม่เคลือบ:


,n_devices
is_coated,
True,44


แยกตามตำแหน่ง (coat_htl/psk = เคลือบทับจริง ไม่รวม additive):


,n_devices
coat_htl,26
coat_psk,22
coat_etl,44


,device_id,htl_mod_kind,psk_mod_kind,coat_htl,coat_psk,coat_etl,is_coated,n_coating_layers
0,10.1002admi.202500231_Sample1,NaN,NaN,False,False,True,True,1
1,10.1002admi.202500231_Sample2,additive,NaN,False,False,True,True,1
2,10.1002admi.202500231_Sample3,additive,NaN,False,False,True,True,1
3,10.1002admi.202500231_Sample4,additive,NaN,False,False,True,True,1
4,10.1016j.jechem.2021.11.029_Sample1,NaN,coating,False,True,True,True,2


## 3. layer 2, 4 ปิดด้วย Annealing จริงไหม

เช็คจาก `process` (`source_sheet == "Process"`) ว่า layer 2 (HTL) และ 4 (PSK) มีขั้น Annealing

In [4]:
_P = pr[pr["source_sheet"] == "Process"]

for lid, name in [(2, "HTL"), (4, "PSK")]:
    g = _P[_P["layer_id"] == lid]
    ann = g[g["process_type"].str.contains("nneal", case=False, na=False)]
    print(f"layer {lid} ({name}): {ann.device_id.nunique()}/{g.device_id.nunique()} devices มี Annealing")

# ธงระดับ device
for lid, col in [(2, "anneal_htl"), (4, "anneal_psk")]:
    hit = _P[(_P["layer_id"] == lid) & _P["process_type"].str.contains("nneal", case=False, na=False)]["device_id"].unique()
    dev[col] = dev["device_id"].isin(hit)

display(dev[["anneal_htl", "anneal_psk"]].sum().rename("n_devices").to_frame())

layer 2 (HTL): 44/44 devices มี Annealing
layer 4 (PSK): 42/44 devices มี Annealing


,n_devices
anneal_htl,44
anneal_psk,42


## 4. แตก `process.parameter` + `unit` เป็นคอลัมน์แยก

`parameter` กับ `unit` เป็นค่าหลายตัวคั่น comma จับคู่ตามตำแหน่ง

```
parameter = "155, 600, 600"   unit = "W, mtorr, sec"
    -> W=155   mtorr=600   sec=600
```

ชื่อคอลัมน์ = `process_type` + `unit` เพราะหน่วยเดียวกันคนละความหมายตามขั้นตอน
(`C` ใน Annealing กับใน Dry คนละเรื่อง)

**⚠️ 9 แถว parameter พิมพ์ผิด** (ตัวเลขติดกัน / comma หาย) — 8 แถวแก้ใน §4.0 ด้วยค่ายืนยันจาก sibling sample ใน Raw Excel, เหลือ 1 แถวกำกวมติดธง `param_note` ไว้ ไม่เดา

### 4.0 แก้ parameter พิมพ์ผิดก่อนแตก

| ดิบ | แก้เป็น | หลักฐาน |
|---|---|---|
| `155600600` | `155, 600, 600` | S3 ไฟล์เดียวกันเขียน `155,600, 600` |
| `1556003600` | `155, 600, 3600` | รูปแบบเดียวกัน plasma นานขึ้น |
| `27010800` | `270, 10800` | S1 เขียน `270, 10800` |
| `603600` | `60, 3600` | S1 เขียน `60, 3600` |
| `80, 0.03 500` | `80, 0.03, 500` | comma หายหลัง 0.03 |

**ไม่แก้**: `100108000` (admi S3, L4 Annealing) — S1=`100, 1800` S4=`100, 10800` ขัดกัน รอเปิด paper ไม่เดา

In [5]:
# ---- §4.0 แก้ parameter พิมพ์ผิด (ยืนยันจาก sibling sample ใน Raw Excel ไฟล์เดียวกัน) ----
# ค่าทั้งหมดของ 10.1002admi.202500231 ยืนยันจาก Supporting Information (admi70014-sup-0001) §1.2
#   "sonicated in acetone followed by isopropanol for 10 minutes each"       -> Wash 1200 s
#   "O2-plasma treatment at 155 W with 600 mtorr O2 ... for 10 min"          -> 155, 600, 600
#   "spin-coating at 3000 rpm for 30 s" (NiOx)                               -> 3000, 30
#   "mixed on a hot plate at 60 C ... during 1 h"                            -> 60, 3600
#   "1000 rpm ... 500 rpm/s ... 10 s / 4000 rpm ... 2000 rpm/s ... 35 s"     -> ตรงอยู่แล้ว
#   "At 10 s before the end ... 0.25 ml of chlorobenzene was dropped"        -> 0.25, -10 (ยืนยัน convention ค่าลบ)
#   "put on a hot plate at 100 C for 30 min"                                 -> 100, 1800  <- ทั้ง 4 sample
PARAM_FIX = {
    # (device_id, layer_id, ค่าดิบ): ค่าที่ถูก
    ("10.1002admi.202500231_Sample1", 1, "155600600"):     "155, 600, 600",   # SI: 155 W, 600 mtorr, 10 min
    ("10.1002admi.202500231_Sample4", 1, "1556003600"):    "155, 600, 600",   # SI: 10 min ทุก sample (ไม่ใช่ 3600 s)
    ("10.1002admi.202500231_Sample3", 2, "27010800"):      "270, 10800",      # ดู NOTE_L2_ANNEAL ด้านล่าง
    ("10.1002admi.202500231_Sample4", 2, "27010800"):      "270, 10800",
    ("10.1002admi.202500231_Sample3", 4, "603600"):        "60, 3600",        # SI: 60 C, 1 h
    ("10.1002admi.202500231_Sample4", 4, "603600"):        "60, 3600",
    # --- PSK annealing: SI ระบุ "100 C for 30 min" = 1800 s เท่ากันทั้ง 4 sample ---
    #     (สูตร perovskite เหมือนกันหมด ต่างแค่ EMIMI ที่ NiOx -- SI ยืนยัน "same batch")
    ("10.1002admi.202500231_Sample2", 4, "100, 108000"):   "100, 1800",
    ("10.1002admi.202500231_Sample3", 4, "100108000"):     "100, 1800",       # เคสที่เคยติดธง -- ปิดแล้ว
    ("10.1002admi.202500231_Sample4", 4, "100, 10800"):    "100, 1800",
    ("10_1021_acsami_4c14300_Sample1", 2, "80, 0.03 500"): "80, 0.03, 500",   # comma หายหลัง 0.03
    ("10_1021_acsami_4c14300_Sample2", 2, "80, 0.03 500"): "80, 0.03, 500",
}
# NOTE_L2_ANNEAL: SI §1.2 (device fabrication) ไม่มีขั้นอบฟิล์ม NiOx เลย -- 270 C 3 h เป็นการ "เผาผง"
#   ใน §1.1 Materials synthesis ("The as-prepared green solid was then annealed at 270 C for 3 h")
#   แถว Annealing ที่ L2 ใน Process sheet จึงซ้ำกับ NiOx synthesis sheet -> htl_anneal_C ของ 4 device นี้
#   ไม่ใช่ขั้นตอนสร้างเซลล์จริง (ดู §9.1 ที่กรองออก)

pr["param_fixed"] = False
_lidn = pd.to_numeric(pr["layer_id"], errors="coerce")
for (d, l, bad), good in PARAM_FIX.items():
    m = (pr["device_id"] == d) & (_lidn == l) & (pr["parameter"].astype(str).str.strip() == bad)
    assert m.sum() == 1, f"PARAM_FIX จับได้ {int(m.sum())} แถว (ต้อง 1): {d} L{l} {bad!r}"
    pr.loc[m, "parameter"] = good
    pr.loc[m, "param_fixed"] = True

print(f"แก้แล้ว {int(pr['param_fixed'].sum())}/11 แถว -- ทุกค่ายืนยันจาก Supporting Information แล้ว ไม่เหลือแถวกำกวม")

แก้แล้ว 11/11 แถว -- ทุกค่ายืนยันจาก Supporting Information แล้ว ไม่เหลือแถวกำกวม


### 4.0b กู้ค่าที่ถูกกรอกไว้ในช่อง Descriptions (audit ข้อ A, B)

In [6]:
# ---- §4.0b กู้ค่าที่ผู้กรอกใส่ไว้ในช่อง Descriptions แทนช่อง Parameter (audit ข้อ A, B) ----
# ตัวแปลงชีตอ่านเฉพาะช่อง parameter ค่าที่อยู่ในช่อง description จึงไม่เคยเข้าระบบ และหายเงียบ
# กวาดไฟล์ Excel ต้นทางครบ 44 ไฟล์แล้วพบ 2 ไฟล์ที่ใช้รูปแบบนี้ รวม 12 แถวที่กู้ได้
#   - solr_202400534_Sample2  1 แถว (ขั้นปั่น 5000 rpm)  -> เดิมทำให้ psk_spin_rpm ผิดเป็น 1000
#   - acsami_5c15868_Sample1  11 แถว (ทั้งชีตใช้รูปแบบนี้) -> เดิมทำให้ RPM/อุณหภูมิของทุกชั้นหาย
# ค่าที่เติมทุกตัวยืนยันจาก sibling sample ในเปเปอร์เดียวกันซึ่งกรอกครบในช่อง parameter
DESC_RECOVER = {
    # (device_id, layer_id, step_order): (parameter, unit)
    ("10_1002_solr_202400534_Sample2", 4, 2): ("5000",           "RPM"),
    ("10_1021_acsami_5c15868_Sample1", 1, 2): ("60",             "C"),
    ("10_1021_acsami_5c15868_Sample1", 2, 1): ("0.2, 0.0006, 25", "Å/s, Pa, C"),
    ("10_1021_acsami_5c15868_Sample1", 2, 2): ("200, 10",        "C, hr"),
    ("10_1021_acsami_5c15868_Sample1", 4, 1): ("2000, 10",       "RPM, sec"),
    ("10_1021_acsami_5c15868_Sample1", 4, 2): ("4000, 30",       "RPM, sec"),
    ("10_1021_acsami_5c15868_Sample1", 4, 3): ("150, 30",        "uL, sec"),
    ("10_1021_acsami_5c15868_Sample1", 4, 4): ("100, 30",        "C, min"),
    ("10_1021_acsami_5c15868_Sample1", 5, 1): ("4000, 30",       "RPM, sec"),
    ("10_1021_acsami_5c15868_Sample1", 5, 2): ("100, 5",         "C, min"),
    ("10_1021_acsami_5c15868_Sample1", 6, 1): ("1400, 30",       "RPM, sec"),
    ("10_1021_acsami_5c15868_Sample1", 7, 1): ("5000, 17",       "RPM, sec"),
}
pr["param_recovered"] = False
_lidn0 = pd.to_numeric(pr["layer_id"], errors="coerce")
_stepn = pd.to_numeric(pr["step_order"], errors="coerce")
_rec_log = []
for (d, l, s), (p_new, u_new) in DESC_RECOVER.items():
    m = (pr["device_id"] == d) & (_lidn0 == l) & (_stepn == s)
    assert m.sum() == 1, f"DESC_RECOVER จับได้ {int(m.sum())} แถว (ต้อง 1): {d} L{l} step{s}"
    i = pr.index[m][0]
    _rec_log.append({"device_id": d, "layer_id": l, "step": s,
                     "description": pr.at[i, "description"],
                     "parameter เดิม": pr.at[i, "parameter"], "unit เดิม": pr.at[i, "unit"],
                     "parameter ใหม่": p_new, "unit ใหม่": u_new})
    pr.loc[m, ["parameter", "unit", "param_recovered"]] = [p_new, u_new, True]

print(f"กู้ค่าจากช่อง description แล้ว {int(pr['param_recovered'].sum())}/{len(DESC_RECOVER)} แถว")
display(pd.DataFrame(_rec_log))

# กันไม่ให้เกิดซ้ำ: ถ้ามีแถวที่ description มีตัวเลข+หน่วย แต่ค่านั้นยังไม่อยู่ใน parameter -> ต้องหยุด
_DESC_QTY = re.compile(r"(?<![\w.])(\d+(?:\.\d+)?)\s*(RPM|rpm|°C|C\b|sec\b|\bs\b|min\b|hr\b)", re.I)
_leak = []
for i, r in pr.iterrows():
    d = str(r["description"]) if pd.notna(r["description"]) else ""
    if not d or pd.isna(r["process_type"]):
        continue
    have = str(r["parameter"]) if pd.notna(r["parameter"]) else ""
    for num, _u in _DESC_QTY.findall(d):
        if num not in have:
            _leak.append({"device_id": r["device_id"], "layer_id": r["layer_id"],
                          "process_type": r["process_type"], "description": d,
                          "parameter": have, "ตัวเลขที่ยังไม่เข้าระบบ": num})
if _leak:
    display(pd.DataFrame(_leak))
assert not _leak, ("มีข้อมูลเชิงปริมาณค้างอยู่ในช่อง description ที่ยังไม่เข้า parameter "
                   "-- เพิ่มลง DESC_RECOVER ก่อนไปต่อ (audit ข้อ A, B)")
print("ด่านตรวจ: ไม่มีข้อมูลเชิงปริมาณตกค้างในช่อง description แล้ว")

กู้ค่าจากช่อง description แล้ว 12/12 แถว


,device_id,layer_id,step,description,parameter เดิม,unit เดิม,parameter ใหม่,unit ใหม่
0,10_1002_solr_202400534_Sample2,4,2,5000 RPM,NaN,NaN,5000,RPM
1,10_1021_acsami_5c15868_Sample1,1,2,60 C,NaN,NaN,60,C
2,10_1021_acsami_5c15868_Sample1,2,1,"E-beam evaporation, rate 0.2 Å/s , 25 C",0.0006,Pa,"0.2, 0.0006, 25","Å/s, Pa, C"
3,10_1021_acsami_5c15868_Sample1,2,2,200 C,10.0,hr,"200, 10","C, hr"
4,10_1021_acsami_5c15868_Sample1,4,1,2000 RPM,10.0,sec,"2000, 10","RPM, sec"
5,10_1021_acsami_5c15868_Sample1,4,2,4000 RPM,30.0,sec,"4000, 30","RPM, sec"
6,10_1021_acsami_5c15868_Sample1,4,3,added at 30 s,150.0,µL,"150, 30","uL, sec"
7,10_1021_acsami_5c15868_Sample1,4,4,100 C,30.0,min,"100, 30","C, min"
8,10_1021_acsami_5c15868_Sample1,5,1,4000 RPM,30.0,sec,"4000, 30","RPM, sec"
9,10_1021_acsami_5c15868_Sample1,5,2,100 C,5.0,min,"100, 5","C, min"


ด่านตรวจ: ไม่มีข้อมูลเชิงปริมาณตกค้างในช่อง description แล้ว


In [7]:
UNIT_FIX = {"µL": "uL", "μL": "uL", "min each": "min"}          # รวมหน่วยพ้อง (รวม mu สองแบบ)

def split_param(param, unit):
    """คืน dict {unit: value} + note ถ้าจับคู่ไม่ได้

    audit ข้อ R/S: เดิมคืน ("", {}) เงียบ ๆ เมื่อไม่มี unit และทิ้งค่าทั้งแถวเมื่อจำนวนไม่ตรง
    ตอนนี้ทุกเส้นทางที่ทำให้ข้อมูลหายต้องติดธงใน param_note เสมอ
    """
    has_p = pd.notna(param) and str(param).strip() != ""
    has_u = pd.notna(unit) and str(unit).strip() != ""
    if not has_p and not has_u:
        return {}, ""                                    # ว่างทั้งคู่ = ช่องว่างของ template ไม่ใช่ข้อมูลหาย
    if has_p and not has_u:
        return {}, f"มี parameter แต่ไม่มี unit: {param!r}"        # เดิมเงียบ -- ตอนนี้ติดธง
    if has_u and not has_p:
        return {}, f"มี unit แต่ไม่มี parameter: {unit!r}"

    ps = [t.strip() for t in str(param).split(",") if t.strip()]
    us = [UNIT_FIX.get(t.strip(), t.strip()) for t in str(unit).split(",") if t.strip()]
    if len(ps) != len(us):
        n = min(len(ps), len(us))                        # เดิมทิ้งทั้งแถว -- ตอนนี้เก็บส่วนที่จับคู่ได้
        out = {}
        for p, u in zip(ps[:n], us[:n]):
            key = u if u not in out else f"{u}#{sum(1 for k in out if k.split('#')[0] == u) + 1}"
            out[key] = pd.to_numeric(p, errors="coerce")
        return out, f"param {len(ps)} ค่า != unit {len(us)} ค่า (เก็บ {n} คู่แรก): {param!r} | {unit!r}"

    out = {}
    for p, u in zip(ps, us):
        key = u if u not in out else f"{u}#{sum(1 for k in out if k.split('#')[0] == u) + 1}"   # หน่วยซ้ำห้ามเขียนทับ
        out[key] = pd.to_numeric(p, errors="coerce")
    return out, ""

parsed, notes = [], []
for _, r in pr.iterrows():
    d, note = split_param(r["parameter"], r["unit"])
    parsed.append(d)
    notes.append(note)

long = []
for i, (d, note) in enumerate(zip(parsed, notes)):
    ptype = pr.iloc[i]["process_type"]
    for u, v in d.items():
        long.append({"row": i, "col": f"{ptype}__{u}", "value": v})
proc_split = pd.DataFrame(long)

pr_out = pr.copy()
pr_out["param_note"] = notes
wide = proc_split.pivot_table(index="row", columns="col", values="value", aggfunc="first")
pr_out = pr_out.join(wide)

n_bad = int((pr_out["param_note"] != "").sum())
n_cols = wide.shape[1]
print(f"แตกเป็น {n_cols} คอลัมน์ | {n_bad} แถวติดธง (param/unit จับคู่ไม่ได้)")
if n_bad:
    display(pr_out.loc[pr_out["param_note"] != "",
                       ["device_id", "layer_id", "process_type", "parameter", "unit", "param_note"]])

แตกเป็น 53 คอลัมน์ | 0 แถวติดธง (param/unit จับคู่ไม่ได้)


### 4.1 ตัวอย่างคอลัมน์ที่แตกได้ — process knob ที่เอาไปใช้ต่อได้

In [8]:
_knob_cols = [c for c in pr_out.columns if "__" in c]
_ns = pr_out["source_sheet"] == "NiOx synthesis"     # สูตรทำผง NiOx ไม่ใช่ขั้นตอนสร้างเซลล์
print("คอลัมน์ทั้งหมด  (!! = มาจาก NiOx synthesis ล้วน -- confound กับ DOI เดียว ห้ามเป็น feature)")
for c in sorted(_knob_cols):
    n = int(pr_out[c].notna().sum())
    if n:
        n_ns = int((pr_out[c].notna() & _ns).sum())
        tag = " !!" if n_ns == n else (f"  ({n_ns} แถวจาก NiOx synthesis)" if n_ns else "")
        print(f"  {c:28s} {n:3d} แถว  ({pr_out[c].min():g} - {pr_out[c].max():g}){tag}")

print("\nตัวอย่าง Spin coating + Annealing:")
_show = [c for c in _knob_cols if c.startswith(("Spin coating", "Annealing"))]
display(pr_out.loc[pr_out[_show].notna().any(axis=1),
                   ["device_id", "layer_id", "process_type", "parameter", "unit"] + _show].head(10))

คอลัมน์ทั้งหมด  (!! = มาจาก NiOx synthesis ล้วน -- confound กับ DOI เดียว ห้ามเป็น feature)
  Annealing__C                 159 แถว  (30 - 450)  (4 แถวจาก NiOx synthesis)
  Annealing__hr                  5 แถว  (3 - 10)  (4 แถวจาก NiOx synthesis)
  Annealing__min                 2 แถว  (5 - 30)
  Annealing__sec               149 แถว  (40 - 36000)
  AntiSolvent__ml                4 แถว  (0.25 - 0.25)
  AntiSolvent__sec              33 แถว  (-10 - 30)
  AntiSolvent__uL               31 แถว  (100 - 300)
  Dip-Coating__sec               4 แถว  (300 - 300)
  Dissolve__M                    4 แถว  (6 - 6) !!
  Dry__C                        11 แถว  (60 - 100)  (4 แถวจาก NiOx synthesis)
  Dry__hr                        4 แถว  (12 - 12) !!
  ElectronBeamEvaporation__C     3 แถว  (25 - 25)
  ElectronBeamEvaporation__Pa    3 แถว  (0.0006 - 0.0006)
  ElectronBeamEvaporation__Å/s   3 แถว  (0.2 - 0.2)
  EthanolWash__RPM               3 แถว  (4000 - 4000)
  EthanolWash__sec               3 แถว  (30 - 4

,device_id,layer_id,process_type,parameter,unit,Annealing__C,Annealing__hr,Annealing__min,Annealing__sec,Spin coating__RPM,Spin coating__RPM/s,Spin coating__sec
3,10.1002admi.202500231_Sample1,2,Spin coating,"3000, 30","RPM, sec",NaN,NaN,NaN,NaN,3000.0,NaN,30.0
4,10.1002admi.202500231_Sample1,2,Annealing,"270, 10800","C, sec",270.0,NaN,NaN,10800.0,NaN,NaN,NaN
6,10.1002admi.202500231_Sample1,4,Spin coating,"1000,500,10","RPM,RPM/s,sec",NaN,NaN,NaN,NaN,1000.0,500.0,10.0
7,10.1002admi.202500231_Sample1,4,Spin coating,"4000,2000,35","RPM,RPM/s,sec",NaN,NaN,NaN,NaN,4000.0,2000.0,35.0
9,10.1002admi.202500231_Sample1,4,Annealing,"100, 1800","C, sec",100.0,NaN,NaN,1800.0,NaN,NaN,NaN
17,10.1002admi.202500231_Sample1,2,Annealing,"270,3","C, hr",270.0,3.0,NaN,NaN,NaN,NaN,NaN
22,10.1002admi.202500231_Sample2,2,Spin coating,"3000, 30","RPM, sec",NaN,NaN,NaN,NaN,3000.0,NaN,30.0
23,10.1002admi.202500231_Sample2,2,Annealing,"270, 10800","C, sec",270.0,NaN,NaN,10800.0,NaN,NaN,NaN
25,10.1002admi.202500231_Sample2,4,Spin coating,"1000,500,10","RPM,RPM/s,sec",NaN,NaN,NaN,NaN,1000.0,500.0,10.0
26,10.1002admi.202500231_Sample2,4,Spin coating,"4000,2000,35","RPM,RPM/s,sec",NaN,NaN,NaN,NaN,4000.0,2000.0,35.0


## 5. สรุป device-level ที่ได้เพิ่ม

รวมธง layer เข้า `dev` — เอาไป join กับ `ml` ใน `after_col_prepare` ได้

In [9]:
cols = ["device_id", "is_coated", "n_coating_layers",
        "coat_htl", "coat_psk", "coat_etl", "anneal_htl", "anneal_psk",
        "htl_mod_kind", "psk_mod_kind"]        # §2.1 coating vs additive
device_layer_flags = dev[cols]
print("device_layer_flags:", device_layer_flags.shape)
display(device_layer_flags.head())
# ยังไม่เขียน -- รวมเขียนทีเดียวใน §10

device_layer_flags: (44, 10)


,device_id,is_coated,n_coating_layers,coat_htl,coat_psk,coat_etl,anneal_htl,anneal_psk,htl_mod_kind,psk_mod_kind
0,10.1002admi.202500231_Sample1,True,1,False,False,True,True,True,NaN,NaN
1,10.1002admi.202500231_Sample2,True,1,False,False,True,True,True,additive,NaN
2,10.1002admi.202500231_Sample3,True,1,False,False,True,True,True,additive,NaN
3,10.1002admi.202500231_Sample4,True,1,False,False,True,True,True,additive,NaN
4,10.1016j.jechem.2021.11.029_Sample1,True,2,False,True,True,True,True,NaN,coating


## 6. normalize เวลาเป็นวินาที + หมายเหตุ anneal temp

หน่วยเวลาปนกัน `sec` / `min` / `hr` รวมเป็น `sec` เดียว (`hr x3600`, `min x60`)

**anneal temp สูง (270-450C) ไม่ใช่บั๊ก** — เป็นการ anneal ชั้น NiOx/HTL (ปกติ 270-450C)
คนละตัวกับ anneal ชั้น PSK (100-150C) **ต้องแยกด้วย `layer_id` ตอนใช้เป็น feature** ห้ามรวม

In [10]:
SEC_PER = {"sec": 1, "min": 60, "hr": 3600}

# รวมเวลาต่อ process_type: <ptype>__time_s = sec + min*60 + hr*3600
_time_units = ["sec", "min", "hr"]
_ptypes = sorted({c.split("__")[0] for c in pr_out.columns if "__" in c})
for pt in _ptypes:
    parts = [(f"{pt}__{u}", SEC_PER[u]) for u in _time_units if f"{pt}__{u}" in pr_out.columns]
    if len(parts) > 1 or (parts and parts[0][0].split("__")[1] != "sec"):
        s = pd.Series(0.0, index=pr_out.index)
        any_val = pd.Series(False, index=pr_out.index)
        for col, mult in parts:
            s = s + pr_out[col].fillna(0) * mult
            any_val = any_val | pr_out[col].notna()
        pr_out[f"{pt}__time_s"] = s.where(any_val)

# ตรวจ anneal แยกตาม layer
_ann = pr_out[pr_out["process_type"] == "Annealing"].dropna(subset=["Annealing__C"])
print("anneal temp แยกตาม layer_id (เห็นว่าชั้นล่าง NiOx/HTL ร้อนกว่า PSK):")
display(_ann.groupby("layer_id")["Annealing__C"].agg(["count", "min", "median", "max"]))

anneal temp แยกตาม layer_id (เห็นว่าชั้นล่าง NiOx/HTL ร้อนกว่า PSK):


,count,min,median,max
layer_id,,,,
1,3,200.0,200.0,200.0
2,55,75.0,150.0,450.0
3,28,60.0,100.0,120.0
4,44,30.0,100.0,105.0
5,10,70.0,100.0,100.0
6,14,60.0,85.0,100.0
7,5,60.0,70.0,70.0


### 6.1 AntiSolvent: รวมหน่วยปริมาตร + convention เวลา

- `__ml` กับ `__uL` เป็น knob เดียวกัน → `AntiSolvent__vol_uL` (0.25 ml = 250 uL)
- เวลาหยด: **ค่าลบ = วินาทีก่อนจบ spin, ค่าบวก = วินาทีหลังเริ่ม** คนละจุดอ้างอิง
  เก็บ `AntiSolvent__timing_ref` กำกับ ค่าดิบห้ามรวมสเกลตรง ๆ
  → §9 แปลงรวมเป็นแกนเดียวแล้ว: `antisolvent_drop_s` = วินาทีนับจากเริ่ม spin ขั้นสุดท้าย
  (ตัวลบแปลงด้วย `psk_spin_last_s + timing`) + `antisolvent_drop_frac` = สัดส่วนของช่วง spin

In [11]:
# ---- §6.1 AntiSolvent: รวมหน่วยปริมาตร + convention เวลา ----
# ปริมาตร: __ml (4 dev) กับ __uL (31 dev) เป็น knob เดียวกัน -> uL เดียว
vol = pd.Series(np.nan, index=pr_out.index)
if "AntiSolvent__uL" in pr_out.columns:
    vol = pr_out["AntiSolvent__uL"].copy()
if "AntiSolvent__ml" in pr_out.columns:
    vol = vol.fillna(pr_out["AntiSolvent__ml"] * 1000)
pr_out["AntiSolvent__vol_uL"] = vol

# เวลา: ค่าลบ = วินาที "ก่อนจบ" spin, ค่าบวก = วินาที "หลังเริ่ม" -- คนละจุดอ้างอิง ห้ามรวมสเกลตรง ๆ
t = pr_out["AntiSolvent__sec"] if "AntiSolvent__sec" in pr_out.columns else pd.Series(np.nan, index=pr_out.index)
pr_out["AntiSolvent__timing_s"] = t
pr_out["AntiSolvent__timing_ref"] = pd.Series(
    np.where(t < 0, "before_end", "after_start"), index=pr_out.index).where(t.notna())

_m = pr_out["AntiSolvent__vol_uL"].notna()
print(f"AntiSolvent__vol_uL: {pr_out.loc[_m, 'device_id'].nunique()} devices "
      f"(uL {int(pr_out['AntiSolvent__uL'].notna().sum())} แถว + ml {int(pr_out['AntiSolvent__ml'].notna().sum())} แถว)")
print("timing_ref:", pr_out["AntiSolvent__timing_ref"].value_counts().to_dict())

AntiSolvent__vol_uL: 35 devices (uL 31 แถว + ml 4 แถว)
timing_ref: {'after_start': 27, 'before_end': 6}


## 7. ฝั่ง Y: เลือก target จาก `properties`

**2 ชั้น แยกกัน**

**Y ประสิทธิภาพ (target หลัก)** — เทรน Voc/Jsc/FF แยกแล้วคูณเป็น PCE
เพราะ `PCE = Voc x Jsc x FF / 100` (พิสูจน์แล้ว corr 0.996) เทรน PCE ตรงไม่ได้ข้อมูลใหม่ แยกได้กลไก

**mediator เชิงกายภาพ** — ใช้ **VTFL** เป็นตัวหลัก (29 dev เยอะกว่า grain_size 13 และ trap_density 23)
VTFL วัดข้อบกพร่องเหมือน trap density แต่เป็นค่าดิบ coverage ดีกว่า

In [12]:
prop = tp["properties"].dropna(subset=["value_parsed"]).copy()
prop["paper"] = prop["device_id"].str.rsplit("_Sample", n=1).str[0]

TARGETS  = ["PCE", "Voc", "Jsc", "FF"]
MEDIATOR = ["VTFL", "Trap density", "Rs", "Rrec", "Contact angle", "Grain size"]

cov = (prop[prop["base_property"].isin(TARGETS + MEDIATOR)]
       .groupby("base_property")
       .agg(rows=("value_parsed", "size"),
            devices=("device_id", "nunique"),
            papers=("paper", "nunique"),
            lo=("value_parsed", "min"), hi=("value_parsed", "max")))
cov["บทบาท"] = cov.index.map(lambda b: "target" if b in TARGETS else "mediator")
cov["ใช้"] = cov["devices"].ge(15) & cov["papers"].ge(5)
display(cov.sort_values(["บทบาท", "devices"], ascending=[True, False]))

# ---- JV: เลือกชุดเดียวกัน (variant + ทิศทางสแกน) ----
# บั๊กเดิมรอบ 1: aggfunc="max" ต่อ property อิสระ -> Champion PCE ปน Average Voc/Jsc/FF
# บั๊กเดิมรอบ 2 (audit ข้อ O): แก้ variant แล้วแต่ยังไม่นับ "ทิศทางสแกน" (RS/FS) เป็นส่วนหนึ่งของชุด
#   RS กับ FS คือการวัดคนละครั้งของเซลล์เดียวกัน ค่าไม่เท่ากัน -- max อิสระจึงยังหยิบข้ามทิศทางได้
#   ตรวจพบ 5/12 device ที่ปนจริง เช่น jechem_Sample1 ได้ FF จาก FS แต่ Voc/Jsc/PCE จาก RS
#   -> คีย์ของชุดต้องเป็น (variant, scan) ไม่ใช่ variant อย่างเดียว
_jv = prop[prop["base_property"].isin(TARGETS)].copy()
_p = _jv["property"].fillna("")
_jv["variant"] = np.where(_p.str.contains("hampion"), "champion",
                 np.where(_p.str.contains("verage"), "average", "plain"))
_jv["scan_k"] = _jv["scan"].fillna("-")                 # "-" = เปเปอร์ไม่ระบุทิศทาง

# ลำดับความสำคัญ: champion ก่อน average, RS ก่อน FS (RS = ค่าที่วงการใช้รายงานเป็นหลัก)
_VAR_ORDER  = ["champion", "plain", "average"]
_SCAN_ORDER = ["-", "RS", "FS"]
_KEYS = [(v, s) for v in _VAR_ORDER for s in _SCAN_ORDER]

piv = {}
for v, s in _KEYS:
    sub = _jv[(_jv["variant"] == v) & (_jv["scan_k"] == s)]
    if len(sub):
        piv[(v, s)] = (sub.pivot_table(index="device_id", columns="base_property",
                                       values="value_parsed", aggfunc="max")
                       .reindex(columns=TARGETS))

rows = []
for dev in sorted(_jv["device_id"].unique()):
    pick, r = None, None
    for k in _KEYS:                                      # ชุดแรกที่มี voc+jsc+ff ครบ "ในทิศทางเดียวกัน"
        if k in piv and dev in piv[k].index and piv[k].loc[dev, ["Voc", "Jsc", "FF"]].notna().all():
            pick, r = k, piv[k].loc[dev]
            break
    if pick is None:
        pick = ("mixed", "-")
        r = (_jv[_jv["device_id"] == dev]
             .pivot_table(index="device_id", columns="base_property", values="value_parsed", aggfunc="max")
             .reindex(columns=TARGETS).loc[dev])
    rows.append({"device_id": dev, "jv_variant": pick[0], "jv_scan": pick[1],
                 **{c.lower(): r[c] for c in TARGETS}})
jv = pd.DataFrame(rows).set_index("device_id")

print("jv_variant:", jv["jv_variant"].value_counts().to_dict())
print("jv_scan   :", jv["jv_scan"].value_counts().to_dict())
_have = jv.dropna(subset=["voc", "jsc", "ff", "pce"])
_err = (_have["voc"] * _have["jsc"] * _have["ff"] / 100 - _have["pce"]).abs()
# เกณฑ์ 0.2: หลังคุมทิศทางแล้ว ส่วนต่างที่เหลือควรเป็นการปัดเศษเท่านั้น (เดิมตั้ง 1.0 ซึ่งหลวมเกินไป)
_over = _err[_err > 0.2]
print(f"เช็ค pce = voc*jsc*ff/100: n={len(_have)} max_err={_err.max():.3f} | เกิน 0.2: {len(_over)} เครื่อง")
if len(_over):
    print("  เครื่องที่ยังเกิน (ต้องอธิบายได้ทุกตัว):")
    display(jv.loc[_over.index, ["jv_variant", "jv_scan", "voc", "jsc", "ff", "pce"]].assign(err=_over.round(3)))

# ---- mediator ----
_med_src = prop[prop["base_property"].isin(["VTFL", "Trap density", "Rs", "Rrec"])].copy()

# audit ข้อ E: Rs หน่วย "Ω/sq" คือความต้านทานแผ่นของฟิล์ม (วัด Hall) ไม่ใช่ series resistance ของเซลล์
#   ค่าต่างกันราว 4,000 เท่า -- แยกออกไปเป็นคอลัมน์ของตัวเอง ห้ามปนกับ rs
_sheet_m = (_med_src["base_property"] == "Rs") & (
    _med_src["unit_corrected"].fillna(_med_src["unit"]).astype(str).str.replace(" ", "").str.lower()
    .str.contains("/sq|persq", regex=True))
sheet_resistance_htl = (_med_src[_sheet_m].groupby("device_id")["value_parsed"].max()
                        .rename("sheet_resistance_htl"))
print(f"\nแยก Rs ที่เป็นความต้านทานแผ่น (Ω/sq) ออก {int(_sheet_m.sum())} แถว: "
      f"{sorted(_med_src.loc[_sheet_m, 'device_id'])}")
_med_src = _med_src[~_sheet_m]

# audit ข้อ H: เดิม aggfunc="median" เฉลี่ย Hole กับ Electron Trap Density รวมกัน ซึ่งเป็นคนละปริมาณ
#   (วัดจาก hole-only กับ electron-only device คนละตัว) และเปเปอร์อื่นรายงานเฉพาะฝั่งโฮล
#   -> ใช้ฝั่งโฮลเป็นหลักให้ตรงชนิดกับเปเปอร์อื่น เก็บฝั่งอิเล็กตรอนเป็นคอลัมน์แยก
_tp_all = _med_src[_med_src["base_property"] == "Trap density"].copy()
_is_e = _tp_all["carrier"].astype(str).str.lower().eq("electron")
_is_h = _tp_all["carrier"].astype(str).str.lower().eq("hole")
trap_density_electron = (_tp_all[_is_e].groupby("device_id")["value_parsed"].max()
                         .rename("trap_density_electron"))
_trap_hole = _tp_all[~_is_e].groupby("device_id")["value_parsed"].max().rename("trap_density")
print(f"trap density: ฝั่งโฮล/ไม่ระบุ {int((~_is_e).sum())} แถว -> trap_density | "
      f"ฝั่งอิเล็กตรอน {int(_is_e.sum())} แถว -> trap_density_electron (แยกไว้ ไม่เฉลี่ยรวม)")

med = (_med_src[_med_src["base_property"] != "Trap density"]
       .pivot_table(index="device_id", columns="base_property", values="value_parsed", aggfunc="median"))
med.columns = [c.lower().replace(" ", "_") for c in med.columns]
med = med.join(_trap_hole, how="outer").join(trap_density_electron, how="outer")

# ---- วินิจฉัยความเทียบได้ของ mediator (audit ข้อ G, F, I) ----
# หลักการ: ค่าที่เปเปอร์คำนวณมาให้ (trap density, Rrec) ขึ้นกับสมมติฐานของแต่ละกลุ่ม
#   จึงเทียบข้ามเปเปอร์ตรง ๆ ไม่ได้ -- แต่แทนที่จะเขียนเตือนไว้ในเอกสารเฉย ๆ
#   ให้คำนวณตัวชี้วัดความน่าสงสัยติดไว้ในไฟล์ คนที่มาใช้ข้อมูลต่อจะเห็นเองโดยไม่ต้องอ่านเอกสาร

# G/F: คำนวณย้อนความหนาฟิล์มจากคู่ (VTFL, trap density) ด้วยสูตร SCLC
#      N_t = 2*eps*eps0*V_TFL / (q*L^2)  ->  L = sqrt(2*eps*eps0*V_TFL / (q*N_t))
#      ถ้าคู่ค่าสอดคล้องกันเอง ต้องได้ความหนาในช่วงจริงของฟิล์ม perovskite (ราว 250-900 nm)
EPS_R, EPS0, Q_E = 32.0, 8.854e-14, 1.602e-19      # eps_r ของ perovskite, F/cm, C
L_OK_NM = (250.0, 900.0)

def _implied_L_nm(vtfl, nt):
    if pd.isna(vtfl) or pd.isna(nt) or nt <= 0:
        return np.nan
    return float(np.sqrt(2 * EPS_R * EPS0 * vtfl / (Q_E * nt)) * 1e7)

_vtfl_dev = (_med_src[_med_src["base_property"] == "VTFL"]
             .groupby("device_id")["value_parsed"].median())

# F: "Defect density" เป็นชื่อที่สามของปริมาณเดียวกัน (2 แถว ของ solr_202200422)
#    เดิมไม่เข้า trap_density เพราะชื่อไม่ตรง -- ตอนนี้ดึงมาไว้ในคอลัมน์ของตัวเอง
#    พร้อมตัวเลขที่อธิบายว่าทำไมถึงไม่รวมเข้า trap_density
_defect = (prop[prop["base_property"] == "Defect density"]
           .groupby("device_id")["value_parsed"].max().rename("defect_density_reported"))

_diag = pd.DataFrame(index=sorted(set(_trap_hole.index) | set(_defect.index)))
_diag["_vtfl"] = _vtfl_dev.reindex(_diag.index)
_diag["_nt"] = _trap_hole.reindex(_diag.index).fillna(_defect.reindex(_diag.index))
_diag["trap_implied_thickness_nm"] = [
    _implied_L_nm(v, n) for v, n in zip(_diag["_vtfl"], _diag["_nt"])]
_L = _diag["trap_implied_thickness_nm"]
_diag["trap_scale_flag"] = np.where(_L.isna(), "no_vtfl",
                            np.where(_L.between(*L_OK_NM), "ok", "scale_suspect"))
trap_diag = _diag[["trap_implied_thickness_nm", "trap_scale_flag"]]

print("\nวินิจฉัย trap density (ความหนาฟิล์มที่คู่ค่าบ่งชี้ -- ของจริง 250-900 nm):")
_show = trap_diag.join(_diag["_nt"].rename("trap_ที่รายงาน")).dropna(subset=["trap_implied_thickness_nm"])
_show["paper"] = [i.rsplit("_Sample", 1)[0][:26] for i in _show.index]
display(_show.groupby("paper").agg(
    n=("trap_scale_flag", "size"),
    ความหนาที่บ่งชี้=("trap_implied_thickness_nm", lambda s: f"{s.min():.0f}-{s.max():.0f} nm"),
    ธง=("trap_scale_flag", lambda s: s.iloc[0])).sort_values("ความหนาที่บ่งชี้"))
print("ธง scale_suspect = ห้ามเทียบ trap density ข้ามเปเปอร์ ให้ใช้ vtfl แทน (วัดตรงจากกราฟ ไม่ผ่านสมมติฐาน)")

# I: Rrec ต่างกันหลายอันดับของขนาดระหว่างเปเปอร์ -- ติดธงอัตโนมัติด้วยระยะห่างจากค่ากลาง (log10)
_rr = _med_src[_med_src["base_property"] == "Rrec"].groupby("device_id")["value_parsed"].median()
if len(_rr):
    _lo = np.log10(_rr.replace(0, np.nan))
    _dev_dex = (_lo - _lo.median()).abs()
    rrec_flag = np.where(_dev_dex > 2, "scale_suspect", "ok")
    rrec_diag = pd.Series(rrec_flag, index=_rr.index, name="rrec_scale_flag")
    _n_sus = int((rrec_diag == "scale_suspect").sum())
    print(f"\nวินิจฉัย Rrec: ค่ากลาง {10**_lo.median():.3g} Ω | "
          f"ห่างเกิน 2 อันดับของขนาด {_n_sus} device -> {sorted(rrec_diag[rrec_diag == 'scale_suspect'].index)}")
else:
    rrec_diag = pd.Series(dtype=object, name="rrec_scale_flag")

Y = jv.join(med).join(_defect).join(trap_diag).join(rrec_diag)
print(f"\nตาราง Y: {Y.shape[0]} devices x {Y.shape[1]} คอลัมน์")
display(Y.describe(include=[np.number]).T.round(3))

,rows,devices,papers,lo,hi,บทบาท,ใช้
base_property,,,,,,,
VTFL,30,29,11,9.000000e-02,1.020000e+00,mediator,True
Trap density,26,23,8,9.900000e+14,1.010000e+17,mediator,True
Rs,22,22,9,5.300000e-01,1.240000e+05,mediator,True
Contact angle,18,18,8,7.000000e+00,8.081000e+01,mediator,True
Rrec,15,15,7,3.720000e+02,1.700000e+08,mediator,True
Grain size,16,13,5,2.830000e+02,1.000000e+03,mediator,False
FF,67,43,15,6.539000e+01,8.468000e+01,target,True
Jsc,67,43,15,1.826000e+01,2.655000e+01,target,True
PCE,74,43,15,1.294000e+01,2.650000e+01,target,True


jv_variant:

 {'champion': 23, 'plain': 16, 'average': 4}
jv_scan   : {'-': 30, 'RS': 13}
เช็ค pce = voc*jsc*ff/100: n=42 max_err=0.503 | เกิน 0.2: 2 เครื่อง
  เครื่องที่ยังเกิน (ต้องอธิบายได้ทุกตัว):


,jv_variant,jv_scan,voc,jsc,ff,pce,err
device_id,,,,,,,
10_1016_j_heliyon_2024_e25352_Sample3,champion,-,1.06,25.00,78.5,20.50,0.302
10_31635_ccschem_025_202505977_Sample1,champion,-,1.15,23.53,76.4,20.17,0.503



แยก Rs ที่เป็นความต้านทานแผ่น (Ω/sq) ออก 2 แถว: ['10_1002_smll_202504237_Sample1', '10_1002_smll_202504237_Sample2']
trap density: ฝั่งโฮล/ไม่ระบุ 23 แถว -> trap_density | ฝั่งอิเล็กตรอน 3 แถว -> trap_density_electron (แยกไว้ ไม่เฉลี่ยรวม)

วินิจฉัย trap density (ความหนาฟิล์มที่คู่ค่าบ่งชี้ -- ของจริง 250-900 nm):


,n,ความหนาที่บ่งชี้,ธง
paper,,,
10_1002adma_202507730,4,179-179 nm,scale_suspect
10_1002_solr_202200422,2,31-32 nm,scale_suspect
10_31635_ccschem_025_20250,3,387-408 nm,ok
10_1016_j_solener_2025_113,2,426-427 nm,ok
10.1016j.nanoen.2021.10628,3,509-756 nm,ok
10_1039_d3lf00258f,4,517-556 nm,ok
10.1016j.jechem.2021.11.02,2,619-620 nm,ok
10_1126_science_ade9637,2,795-795 nm,ok


ธง scale_suspect = ห้ามเทียบ trap density ข้ามเปเปอร์ ให้ใช้ vtfl แทน (วัดตรงจากกราฟ ไม่ผ่านสมมติฐาน)

วินิจฉัย Rrec: ค่ากลาง 2.23e+04 Ω | ห่างเกิน 2 อันดับของขนาด 2 device -> ['10_1021_acsami_4c14300_Sample1', '10_1021_acsami_4c14300_Sample2']

ตาราง Y: 43 devices x 15 คอลัมน์


,count,mean,std,min,25%,50%,75%,max
pce,42.0,1.995400e+01,3.622000e+00,1.294000e+01,1.700500e+01,2.016500e+01,2.279800e+01,2.650000e+01
voc,43.0,1.099000e+00,7.400000e-02,9.700000e-01,1.040000e+00,1.110000e+00,1.152000e+00,1.230000e+00
jsc,43.0,2.313600e+01,2.136000e+00,1.826000e+01,2.136500e+01,2.349000e+01,2.471500e+01,2.655000e+01
ff,43.0,7.769100e+01,4.694000e+00,6.539000e+01,7.510000e+01,7.846000e+01,8.119000e+01,8.468000e+01
rrec,15.0,1.278550e+07,4.384244e+07,3.720000e+02,3.652000e+03,2.226000e+04,3.778508e+04,1.700000e+08
rs,20.0,1.354800e+01,1.356500e+01,5.300000e-01,9.150000e-01,8.970000e+00,2.509200e+01,4.211000e+01
vtfl,28.0,4.580000e-01,2.690000e-01,9.000000e-02,2.280000e-01,4.150000e-01,6.650000e-01,1.020000e+00
trap_density,22.0,1.301977e+16,2.378321e+16,9.900000e+14,2.387250e+15,3.795000e+15,6.002500e+15,1.010000e+17
trap_density_electron,3.0,3.020000e+15,9.585927e+14,2.270000e+15,2.480000e+15,2.690000e+15,3.395000e+15,4.100000e+15
defect_density_reported,2.0,2.335000e+18,5.161880e+17,1.970000e+18,2.152500e+18,2.335000e+18,2.517500e+18,2.700000e+18


## 8. Stability: คิด index ก่อนใช้เป็น Y

retention ดิบใช้ตรงไม่ได้ — วัดคนละสภาพ (25-85C, 1-5000 ชม.)

ทำ 2 อย่าง
1. `degradation_rate` = `(100 - retention) / time_hr` [%/ชม.] — normalize เวลา
2. `temp_bin` — จับกลุ่มอุณหภูมิ ใช้เทียบเฉพาะในกลุ่มเดียวกัน

**ข้อจำกัด**: การเสื่อมไม่เชิงเส้น (burn-in เร็วแล้วช้า) rate เทียบข้าม timescale ตรง ๆ ไม่ได้
ใช้ retention @ 25C bin เป็น Y เสถียร (14 dev) เป็นทางที่สะอาดสุด แต่ยัง exploratory

In [13]:
stab = tp["stability"].copy()
stab["ret"] = pd.to_numeric(stab["retention_pct_parsed"], errors="coerce")
stab["t_hr"] = pd.to_numeric(stab["time_hr"], errors="coerce")
stab["Tp"] = pd.to_numeric(stab["temperature_c_parsed"], errors="coerce")
stab = stab.dropna(subset=["ret"])

stab["degradation_rate"] = (100 - stab["ret"]) / stab["t_hr"]         # %/hr
stab["temp_bin"] = pd.cut(stab["Tp"], [0, 30, 60, 100],
                          labels=["~25C", "45-50C", "65-85C"])

print("index รายกลุ่มอุณหภูมิ:")
display(stab.groupby("temp_bin", observed=True).agg(
    devices=("device_id", "nunique"),
    ret_median=("ret", "median"),
    rate_median=("degradation_rate", "median")).round(3))
print(f"ไม่มี temp: {int(stab['Tp'].isna().sum())} แถว (ใช้ได้เฉพาะ degradation_rate)")

# Y เสถียร: retention ที่ ~25C bin (สภาพใกล้กันสุด)
stab_Y = (stab[stab["temp_bin"] == "~25C"]
          .groupby("device_id").agg(retention_25c=("ret", "max"),
                                    degradation_rate=("degradation_rate", "min")))
print(f"\nY เสถียร (retention @ 25C): {len(stab_Y)} devices")
display(stab_Y.head())

index รายกลุ่มอุณหภูมิ:


,devices,ret_median,rate_median
temp_bin,,,
~25C,14,82.355,0.018
45-50C,4,65.000,0.041
65-85C,13,75.000,0.023


ไม่มี temp: 14 แถว (ใช้ได้เฉพาะ degradation_rate)

Y เสถียร (retention @ 25C): 14 devices


,retention_25c,degradation_rate
device_id,,
10.1021_acsami.5c16972_Sample1,43.0,0.040714
10.1021_acsami.5c16972_Sample2,72.0,0.020000
10.1021_acsami.5c16972_Sample3,90.0,0.007143
10_1002_solr_202200422_Sample1,62.0,0.015833
10_1002_solr_202200422_Sample2,92.0,0.001600


## 8.5 PSK composition (a_Cs / a_FA / x_Br)

`column_prepare` §J คำนวณครบ 44 devices แต่**ไม่เคย export ลงดิสก์** — port ตรรกะเดิมทุกบรรทัด (J1-J5, TOL=0.03) มาคำนวณใหม่จากตาราง prepared ที่นี่

สูตรนามเป็นหลัก → precursor เติมช่องว่าง + cross-check | export เป็น `psk_composition.csv` ใน §10

In [14]:
# ---- §8.5 psk_composition -- port ตรรกะเดิมจาก column_prepare §J (J1-J5) ----
# column_prepare คำนวณแล้วแต่ไม่เคยเขียนดิสก์ -- คำนวณใหม่จากตาราง prepared ที่นี่แล้ว export ใน §10

# J1: อ่านสัดส่วนจากสูตรในชื่อชั้น
CATIONS = ["FA", "MA", "Cs", "Rb", "Pb", "Sn", "Br", "Cl", "I", "K"]   # ยาว->สั้น กัน FA ถูกอ่านเป็น F+A
TOKEN = re.compile("(" + "|".join(CATIONS) + r")(\d*\.?\d+)")

def _site_from_str(part):
    out = {}
    for inner, scale in re.findall(r"\(([^)]*)\)\s*(\d*\.?\d+)?", part):
        k = float(scale) if scale else 1.0
        for ion, frac in TOKEN.findall(inner):
            out[ion] = out.get(ion, 0) + float(frac) * k
    rest = re.sub(r"\([^)]*\)\s*\d*\.?\d*", "", part)
    for ion, frac in TOKEN.findall(rest):
        out[ion] = out.get(ion, 0) + float(frac)
    return out

def composition_from_formula(f):
    s = str(f).strip().strip("[]").strip()
    m = re.search(r"(Pb|Sn)", s)
    if not m:
        return None
    A, X = _site_from_str(s[:m.start()]), _site_from_str(s[m.end():])
    sA, sX = sum(A.values()), sum(X.values())
    if not sA or not sX:
        return None
    return ({k: v / sA for k, v in A.items()}, {k: v / sX for k, v in X.items()}, m.group(1))

# J2: molar mass + ไอออนที่สารตั้งต้นให้
MW = {"CsI": 259.81, "CsBr": 212.81, "FAI": 171.97, "MAI": 158.97, "MABr": 111.97,
      "MACl": 67.52, "PbI2": 461.01, "PbBr2": 367.01, "PbCl2": 278.11}
ION = {"CsI":  {"A": {"Cs": 1}, "X": {"I": 1}},   "CsBr": {"A": {"Cs": 1}, "X": {"Br": 1}},
       "FAI":  {"A": {"FA": 1}, "X": {"I": 1}},   "MAI":  {"A": {"MA": 1}, "X": {"I": 1}},
       "MABr": {"A": {"MA": 1}, "X": {"Br": 1}},  "MACl": {"A": {"MA": 1}, "X": {"Cl": 1}},
       "PbI2": {"B": {"Pb": 1}, "X": {"I": 2}},   "PbBr2":{"B": {"Pb": 1}, "X": {"Br": 2}},
       "PbCl2":{"B": {"Pb": 1}, "X": {"Cl": 2}}}
MOLAR = {"m": 1.0, "mol": 1.0, "m total": 1.0, "mmol": 0.001}
MASS  = {"mg": 0.001, "g": 1.0, "g / 2ml": 1.0}
VOLATILE = {"MACl"}

def canon_precursor(s):
    k = re.sub(r"[^a-z0-9]", "", str(s).lower())
    for n in MW:
        if re.sub(r"[^a-z0-9]", "", n.lower()) == k:
            return n
    return None

def unit_family(u):
    u = str(u).strip().lower()
    return "molar" if u in MOLAR else ("mass" if u in MASS else None)

# J3: join ชั้น PSK กับสารตั้งต้น
psk = lay.loc[lay["layer_type"] == "PSK",
              ["device_id", "layer_id", "material"]].rename(columns={"material": "layer_material"})
jm = psk.merge(tp["materials"], on=["device_id", "layer_id"], how="left")
jm["prec"] = jm["material"].map(canon_precursor)
jm = jm.dropna(subset=["prec"]).copy()
jm = jm[~jm["prec"].isin(VOLATILE)]
jm["ufam"] = jm["unit"].map(unit_family)
jm["amt"]  = pd.to_numeric(jm["amount"], errors="coerce")

# J4: คำนวณจากสารตั้งต้นทีละ device
prec_comp, prec_status = {}, {}
for dev, g in jm.groupby("device_id"):
    fams = set(g["ufam"].dropna())
    bad  = g[g["ufam"].isna() | g["amt"].isna()]
    if len(bad):
        prec_status[dev] = "ตกหล่น: " + ", ".join(sorted(set(bad["prec"])))
    elif len(fams) != 1:
        prec_status[dev] = f"หน่วยปนตระกูล {sorted(fams)}"
    elif not any(ION[p].get("B") for p in g["prec"]):
        prec_status[dev] = "ไม่มี B-site (สูตรไม่ครบ)"
    else:
        fam, A, X = fams.pop(), {}, {}
        conv = MOLAR if fam == "molar" else MASS
        for _, r in g.iterrows():
            mol = r["amt"] * conv[str(r["unit"]).strip().lower()]
            if fam == "mass":
                mol /= MW[r["prec"]]
            spec = ION[r["prec"]]
            for i, mlt in spec.get("A", {}).items(): A[i] = A.get(i, 0) + mol * mlt
            for i, mlt in spec.get("X", {}).items(): X[i] = X.get(i, 0) + mol * mlt
        sA, sX = sum(A.values()), sum(X.values())
        prec_comp[dev] = ({k: v/sA for k, v in A.items()}, {k: v/sX for k, v in X.items()}, "Pb")
        prec_status[dev] = f"ok ({fam})"

# J5: รวม 2 แหล่ง สูตรเป็นหลัก precursor เติมช่องว่าง + cross-check
TOL = 0.03
rows = []
for _, r in psk.iterrows():
    dev = r["device_id"]
    f_res, p_res = composition_from_formula(r["layer_material"]), prec_comp.get(dev)
    row = {"device_id": dev, "layer_material": r["layer_material"],
           "precursor_status": prec_status.get(dev, "ไม่มีแถว precursor")}
    if f_res:
        A, X, b = f_res
        row["comp_source"] = "formula"
        if p_res:
            pA, pX, _ = p_res
            diff = [k for k in set(A) | set(X) | set(pA) | set(pX)
                    if abs({**A, **X}.get(k, 0) - {**pA, **pX}.get(k, 0)) > TOL]
            row["cross_check"] = "ตรง" if not diff else "ต่าง: " + ", ".join(sorted(diff))
    elif p_res:
        A, X, b = p_res
        row["comp_source"] = "precursor"
        row["cross_check"] = "-- (สูตรแตกไม่ได้)"
    else:
        row["comp_source"] = "ไม่มีข้อมูล"
        rows.append(row)
        continue
    row["b_site"] = b
    row.update({f"a_{k}": round(v, 3) for k, v in A.items()})
    row.update({f"x_{k}": round(v, 3) for k, v in X.items()})
    rows.append(row)

psk_composition = pd.DataFrame(rows)
front = ["device_id", "comp_source", "cross_check", "b_site"]
psk_composition = psk_composition[[c for c in front if c in psk_composition.columns]
                                  + [c for c in psk_composition.columns if c not in front]]

n_ok = int((psk_composition["comp_source"] != "ไม่มีข้อมูล").sum())
print(f"องค์ประกอบ {n_ok}/{psk_composition['device_id'].nunique()} devices | "
      f"source: {psk_composition['comp_source'].value_counts().to_dict()}")

# device-level สำหรับ merge เข้า target_table (ตามแผน §9: a_Cs / a_FA / x_Br)
# ไอออนที่ไม่อยู่ในสูตร = สัดส่วน 0 จริง ไม่ใช่ missing (เช่น pure iodide -> x_Br = 0)
# fill 0 เฉพาะ device ที่คำนวณองค์ประกอบได้ -- ที่คำนวณไม่ได้คง NaN
_cd = psk_composition.drop_duplicates("device_id").set_index("device_id")
comp_dev = _cd.reindex(columns=["a_Cs", "a_FA", "x_Br"])
_ok = _cd["comp_source"] != "ไม่มีข้อมูล"
comp_dev.loc[_ok] = comp_dev.loc[_ok].fillna(0.0)
print("coverage (หลัง fill 0 ให้ไอออนที่ไม่อยู่ในสูตร):", comp_dev.notna().sum().to_dict())

องค์ประกอบ 44/44 devices | source: {'formula': 42, 'precursor': 2}
coverage (หลัง fill 0 ให้ไอออนที่ไม่อยู่ในสูตร): {'a_Cs': 44, 'a_FA': 44, 'x_Br': 44}


## 8.6 `material_family` ที่เป็น NaN — จัดการตามบทบาท ไม่ใช่เติมมั่ว

NaN ในแต่ละตาราง/ชั้นคนละความหมาย:

| ที่ | สถานะจริง | จัดการ |
|---|---|---|
| layers L4 (PSK) | NaN ทั้งหมด — material เป็นสูตรเคมี | มีตัวแทนที่ดีกว่าแล้ว: `psk_composition` (§8.5) |
| layers L1 / L2 / L8 | TCO / NiOx / Metal ล้วน — **variance = 0** | ใช้เป็น feature ไม่ได้ ปล่อยไว้ |
| layers L6 (ETL) | PCBM 24 / C60 18 / BCP 2 — **มี variance**, PCBM 9 paper / C60 5 paper | **ดึงเป็น `etl_family`** เข้า target_table (cell ล่าง) |
| materials | precursor / solvent / additive (CsI, FAI, DMF, ...) | บทบาทมีในคอลัมน์ `type` อยู่แล้ว — family ไม่ applicable |
| vendors | provenance | ปล่อย NaN ไม่ใช่ feature |

In [15]:
# ---- §8.6 etl_family: ตัวเดียวใน material_family ที่มี variance พอเป็น feature ----
_etl = lay[(lay["layer_id"] == 6) & lay["material"].notna()]
assert (_etl.groupby("device_id").size() == 1).all(), "L6 มีหลายแถวต่อ device -- ต้องรวมก่อน"

arch_dev = _etl.set_index("device_id")[["material_family"]].rename(columns={"material_family": "etl_family"})
arch_dev["etl_family"] = arch_dev["etl_family"].fillna(_etl.set_index("device_id")["material"])

_ap = pd.Series(arch_dev.index, index=arch_dev.index).str.rsplit("_Sample", n=1).str[0]
print("etl_family:", arch_dev["etl_family"].value_counts(dropna=False).to_dict())
print("กระจาย paper:", {f: int(_ap[arch_dev["etl_family"] == f].nunique()) for f in arch_dev["etl_family"].unique()},
      "-- PCBM/C60 ข้ามหลาย paper ใช้ใต้ LOPO ได้ | BCP 2 dev/1 paper ระวังตอนตีความ")

etl_family: {'PCBM': 24, 'C60': 18, 'BCP': 2}
กระจาย paper: {'C60': 5, 'PCBM': 9, 'BCP': 1} -- PCBM/C60 ข้ามหลาย paper ใช้ใต้ LOPO ได้ | BCP 2 dev/1 paper ระวังตอนตีความ


## 8.7 ปิดช่องโหว่ `value_parsed` ไม่ตาม `value_after_unit_fix`

`column_prepare` §3.2 คำนวณ `value_parsed = to_numeric(value)` จาก**ค่าดิบ** ไม่เคยอ่าน `value_after_unit_fix`
ผล: การแก้ที่**เปลี่ยนตัวเลข** (ไม่ใช่แค่เปลี่ยนป้าย unit) หายไปเงียบ ๆ จากคอลัมน์ที่ทุกคนใช้

ตรวจ `unit_value_errors.csv` (27 แถว) ทีละกลุ่ม:

| กลุ่ม | จำนวน | สถานะ |
|---|---|---|
| Voc/Jsc ติดป้าย `%` | 12 | ✅ แก้แล้ว — เปลี่ยนแค่ป้าย unit ตัวเลขไม่เปลี่ยน `value_parsed` ถูกอยู่แล้ว |
| FF เป็นเศษส่วน | 8 | ✅ แก้แล้ว — ช่วง 3 คูณ 100 เข้า `value_parsed` โดยตรง |
| Hole mobility หน่วย m² | 3 | ⚠️ **แก้ครึ่งเดียว** — `value_after_unit_fix` = ×1e4 ถูก แต่ `value_parsed` ยังเป็นค่า m² เดิม |
| (false positive จาก regex เก่า) | 4 | หน่วยเป็น `cm2 V-1 s-1` อยู่แล้ว ไม่ต้องแก้ |

เซลล์ล่างซิงก์ `value_parsed` ให้ตาม `value_after_unit_fix` ทุกแถวที่ตัวเลขต่างกัน

In [16]:
# ---- §8.7 ซิงก์ value_parsed ให้ตาม value_after_unit_fix (เขียนกลับเข้า tp["properties"]) ----
_pp = tp["properties"]
_v  = pd.to_numeric(_pp["value"], errors="coerce")
_vc = pd.to_numeric(_pp["value_after_unit_fix"], errors="coerce")
_vp = pd.to_numeric(_pp["value_parsed"], errors="coerce")

# แถวที่การแก้เปลี่ยน "ตัวเลข" จริง (ไม่ใช่แค่เปลี่ยนป้าย unit) แล้ว value_parsed ยังไม่ตาม
_need = (_vc.notna() & _v.notna() & ~np.isclose(_vc, _v, rtol=1e-9)
         & _vp.notna() & ~np.isclose(_vp, _vc, rtol=1e-9))
print(f"แถวที่ value_parsed ยังไม่ตาม value_after_unit_fix: {int(_need.sum())}")

# §7 (Y) รันไปก่อนหน้าแล้ว -- ต้องพิสูจน์ว่าไม่มีแถวไหนเป็น target/mediator ที่ใช้จริง
_used = set(TARGETS) | {"VTFL", "Trap density", "Rs", "Rrec"}
assert not _pp.loc[_need, "base_property"].isin(_used).any(),     "มีแถว target/mediator ถูกแก้ -- ต้องย้าย §8.7 ไปก่อน §7 แล้วรันใหม่"

if _need.any():
    display(_pp.loc[_need, ["device_id", "property", "value", "unit",
                            "unit_corrected", "value_after_unit_fix", "value_parsed"]])
    _pp.loc[_need, "value_parsed"] = _vc[_need]
    _pp.loc[_need, "value_note"] = (
        _pp.loc[_need, "value_note"].fillna("")
        + " | recheck §8.7: ซิงก์ value_parsed = value_after_unit_fix (m2 -> cm2 x1e4)").str.strip(" |")
    print("ซิงก์แล้ว -- ค่าใหม่:")
    display(_pp.loc[_need, ["device_id", "property", "unit_corrected", "value_parsed"]])

# gate: ไม่เหลือแถวที่ value_parsed ขัดกับ value_after_unit_fix
_chk = pd.to_numeric(_pp["value_parsed"], errors="coerce")
assert not (_vc.notna() & _v.notna() & ~np.isclose(_vc, _v, rtol=1e-9)
            & _chk.notna() & ~np.isclose(_chk, _vc, rtol=1e-9)).any(), "ยังมี value_parsed ไม่ตาม value_after_unit_fix"
print("gate ผ่าน: value_parsed สอดคล้องกับ value_after_unit_fix ทุกแถว")

แถวที่ value_parsed ยังไม่ตาม value_after_unit_fix: 3


,device_id,property,value,unit,unit_corrected,value_after_unit_fix,value_parsed
123,10.1021_acsami.5c16972_Sample1,Hole mobility,0.000367,m2 V-1 s-1,cm2 V-1 s-1,3.671,0.000367
143,10.1021_acsami.5c16972_Sample2,Hole mobility,0.000465,m2 V-1 s-1,cm2 V-1 s-1,4.646,0.000465
163,10.1021_acsami.5c16972_Sample3,Hole mobility,0.000694,m2 V-1 s-1,cm2 V-1 s-1,6.941,0.000694


ซิงก์แล้ว -- ค่าใหม่:


,device_id,property,unit_corrected,value_parsed
123,10.1021_acsami.5c16972_Sample1,Hole mobility,cm2 V-1 s-1,3.671
143,10.1021_acsami.5c16972_Sample2,Hole mobility,cm2 V-1 s-1,4.646
163,10.1021_acsami.5c16972_Sample3,Hole mobility,cm2 V-1 s-1,6.941


gate ผ่าน: value_parsed สอดคล้องกับ value_after_unit_fix ทุกแถว


### 8.8 แก้ `PSK optical band gap` จาก Figure S10b (SI)

ค่าที่กรอกไว้ของ Sample2/3/4 = 1.620 eV มาจาก **PL peak** ในตัวบท ("maximum PL ... now at 1.62 eV")
แต่คอลัมน์นี้คือ **optical band gap จาก Tauc plot** ซึ่ง SI Figure S10b ให้ค่ารายตัว:

| device | EMIMI | Tauc Eg (SI) | ที่กรอกไว้ |
|---|---|---|---|
| Sample1 | control | 1.609 | 1.609 ✅ |
| Sample2 | 1% | **1.605** | 1.620 ❌ |
| Sample3 | 2% | **1.615** | 1.620 ❌ |
| Sample4 | 4% | **1.616** | 1.620 ❌ |

In [17]:
# ---- §8.8 แก้ PSK optical band gap ด้วยค่าจาก SI Figure S10b (Tauc plot) ----
PROP_FIX = {   # (device_id, property): ค่าที่ถูก
    ("10.1002admi.202500231_Sample2", "PSK optical band gap"): 1.605,
    ("10.1002admi.202500231_Sample3", "PSK optical band gap"): 1.615,
    ("10.1002admi.202500231_Sample4", "PSK optical band gap"): 1.616,
}
_pp = tp["properties"]
_n = 0
for (d, prop_name), good in PROP_FIX.items():
    m = (_pp["device_id"] == d) & (_pp["property"] == prop_name)
    assert m.sum() == 1, f"PROP_FIX จับได้ {int(m.sum())} แถว (ต้อง 1): {d} / {prop_name}"
    _pp.loc[m, "value_parsed"] = good
    _pp.loc[m, "value_note"] = (_pp.loc[m, "value_note"].fillna("")
        + " | recheck §8.8: ค่าเดิมเป็น PL peak แก้เป็น Tauc Eg จาก SI Figure S10b").str.strip(" |")
    _n += 1
print(f"แก้ optical band gap {_n} แถว (ยืนยันจาก SI Figure S10b)")
display(_pp[(_pp.device_id.str.startswith("10.1002admi")) & (_pp.property == "PSK optical band gap")]
        [["device_id", "value", "value_parsed", "value_note"]])

แก้ optical band gap 3 แถว (ยืนยันจาก SI Figure S10b)


,device_id,value,value_parsed,value_note
7,10.1002admi.202500231_Sample1,1.609,1.609,NaN
19,10.1002admi.202500231_Sample2,1.620,1.605,recheck §8.8: ค่าเดิมเป็น PL peak แก้เป็น Tauc...
31,10.1002admi.202500231_Sample3,1.620,1.615,recheck §8.8: ค่าเดิมเป็น PL peak แก้เป็น Tauc...
43,10.1002admi.202500231_Sample4,1.620,1.616,recheck §8.8: ค่าเดิมเป็น PL peak แก้เป็น Tauc...


### 8.8b แก้ property: เครื่องหมาย VBM, ชื่อที่ชนกัน, ชั้นของแถบพลังงาน

In [18]:
# ---- §8.8b แก้ property ตามผลการตรวจสอบย้อนกลับ (audit ข้อ C, D, J) ----
_pp = tp["properties"]

# ข้อ C: VBM ของ ccschem_Sample1 เป็น +5.02 -- เครื่องหมายลบหาย
#   หลักฐาน 3 ชั้น: (1) ระดับพลังงานอ้างอิงสุญญากาศต้องเป็นลบเสมอ
#   (2) Sample2/3 ของเปเปอร์เดียวกัน จากรูปเดียวกัน (Figure S8) เป็น -5.53 และ -5.63
#   (3) -5.02 ให้ลำดับ NiOx เปล่า -> เคลือบ SAM แล้วลึกลง ซึ่งเป็นทิศทางมาตรฐานและเป็นข้ออ้างของเปเปอร์เอง
SIGN_FIX = {("10_31635_ccschem_025_202505977_Sample1", "VBM"): -5.02}
for (d, pname), good in SIGN_FIX.items():
    m = (_pp["device_id"] == d) & (_pp["property"] == pname)
    assert m.sum() == 1, f"SIGN_FIX จับได้ {int(m.sum())} แถว (ต้อง 1): {d} / {pname}"
    _pp.loc[m, ["value_parsed", "value_after_unit_fix"]] = good
    _pp.loc[m, "value_note"] = (_pp.loc[m, "value_note"].fillna("")
        + " | recheck §8.8b: เครื่องหมายลบหายในต้นฉบับ แก้ +5.02 -> -5.02 (audit ข้อ C)").str.strip(" |")

# ข้อ D: ชื่อ "VBM" ถูกใช้กับ 2 ปริมาณที่ต่างกัน
#   ค่าลบ ~ -5 eV = ระดับ VBM อ้างอิงสุญญากาศ
#   ค่าบวก 0.4-0.6 eV = ระยะจากระดับ Fermi ถึงขอบแถบเวเลนซ์ (E_F - E_VBM) ซึ่งเป็นบวกตามนิยาม
#   ตรวจไขว้: smll_Sample1 work function 4.39 + 0.55 = -4.94 eV เข้าช่วงปกติพอดี -> ค่าถูก แต่ชื่อกำกวม
_vbm = (_pp["base_property"] == "VBM") & pd.to_numeric(_pp["value_parsed"], errors="coerce").gt(0)
_pp.loc[_vbm, "base_property"] = "VBM offset from EF"
_pp.loc[_vbm, "value_note"] = (_pp.loc[_vbm, "value_note"].fillna("")
    + " | recheck §8.8b: เป็น E_F - E_VBM (บวก) คนละปริมาณกับ VBM อ้างอิงสุญญากาศ (audit ข้อ D)").str.strip(" |")
print(f"แยกชื่อ VBM ที่เป็นคนละปริมาณ: {int(_vbm.sum())} แถว -> 'VBM offset from EF'")
display(_pp.loc[_pp["base_property"].isin(["VBM", "VBM offset from EF"]),
                ["device_id", "base_property", "value_parsed"]].sort_values("value_parsed"))

# ข้อ J: "Bandgap" 3.80 eV ของ solener คือแถบพลังงานของ NiOx ไม่ใช่ perovskite แต่ไม่ได้ระบุชั้น
#   เทียบกับ admi ที่ตั้งชื่อแยกชัดเจนเป็น "HTL optical band gap" / "PSK optical band gap"
_bg = (_pp["base_property"].isin(["Bandgap", "optical band gap", "Energy gap"])
       & pd.to_numeric(_pp["value_parsed"], errors="coerce").gt(2.5) & _pp["layer"].isna())
_pp.loc[_bg, "layer"] = "HTL"
_pp.loc[_bg, "value_note"] = (_pp.loc[_bg, "value_note"].fillna("")
    + " | recheck §8.8b: ค่า > 2.5 eV เป็นของ HTL ไม่ใช่ perovskite เติม layer ให้ (audit ข้อ J)").str.strip(" |")
print(f"เติม layer=HTL ให้ค่าแถบพลังงานที่ไม่ระบุชั้นและสูงเกิน perovskite: {int(_bg.sum())} แถว")

แยกชื่อ VBM ที่เป็นคนละปริมาณ: 3 แถว -> 'VBM offset from EF'


,device_id,base_property,value_parsed
8,10.1002admi.202500231_Sample1,VBM,-5.72
560,10_31635_ccschem_025_202505977_Sample3,VBM,-5.63
165,10.1021_acsami.5c16972_Sample3,VBM,-5.61
145,10.1021_acsami.5c16972_Sample2,VBM,-5.58
544,10_31635_ccschem_025_202505977_Sample2,VBM,-5.53
125,10.1021_acsami.5c16972_Sample1,VBM,-5.44
20,10.1002admi.202500231_Sample2,VBM,-5.42
32,10.1002admi.202500231_Sample3,VBM,-5.42
44,10.1002admi.202500231_Sample4,VBM,-5.42
84,10.1016j.jechem.2021.11.029_Sample2,VBM,-5.29


เติม layer=HTL ให้ค่าแถบพลังงานที่ไม่ระบุชั้นและสูงเกิน perovskite: 1 แถว


### 8.9 แก้ `CsI 0.7 M` ของ `solr_202200422` เป็น `0.07 M` (typo ในตัวเปเปอร์เอง)

Raw Excel กรอก **ตรงตามเปเปอร์** (SI: *"0.7 M cesium iodide (CsI), 1.30 M PbI₂, 1.19 M FAI, 0.21 M PbBr₂, 0.21 M MABr"*)
— แต่หลักฐานฟิสิกส์ 4 ชั้นชี้ว่า **เปเปอร์พิมพ์ผิดจาก 0.07**:

1. ถ้า 0.07 → a_Cs=0.048, a_FA=0.81, a_MA=0.14 = สูตรมาตรฐาน + **A/B = 0.97 stoichiometric** / ถ้า 0.7 → A เกิน B 39% ผิดธรรมชาติ
2. ตัวอื่นทั้งหมดตรง **recipe Saliba คลาสสิก scale ×1.18** ซึ่ง CsI effective ~0.07-0.09
3. **PL peak ~770 nm** (SI Fig S12) = bandgap มาตรฐาน — Cs 33% จริงต้อง blue-shift ชัด
4. เปเปอร์เรียกเองว่า "CsFAMA triple cation" (= Cs ~5% ตาม convention)

user ยืนยันให้แก้ 2026-08-04

In [19]:
# ---- §8.9 แก้ CsI 0.7 -> 0.07 M (typo ในเปเปอร์ solr_202200422 -- หลักฐานใน markdown ด้านบน) ----
_mm = tp["materials"]
_m = (_mm["device_id"].str.startswith("10_1002_solr_202200422")
      & (_mm["material"] == "CsI") & (_mm["layer_id"] == 4))
assert int(_m.sum()) == 2, f"คาด 2 แถว เจอ {int(_m.sum())}"
assert (pd.to_numeric(_mm.loc[_m, "amount"]) == 0.7).all(), "ค่าดิบไม่ใช่ 0.7 แล้ว -- เช็คก่อนแก้"
_mm.loc[_m, "amount"] = 0.07
print("แก้ CsI 0.7 -> 0.07 M:", int(_m.sum()), "แถว (Sample1, Sample2)")

# คำนวณ composition ใหม่เฉพาะ 2 device นี้ (ตรรกะเดียวกับ §8.5 -- rerun ทั้ง notebook อยู่แล้ว
# เซลล์นี้อยู่ก่อน §8.5? ไม่ -- §8.5 รันไปแล้ว จึงต้อง patch psk_composition + comp_dev ตรงนี้)
for dev in ["10_1002_solr_202200422_Sample1", "10_1002_solr_202200422_Sample2"]:
    g = jm[jm["device_id"] == dev].copy()
    g.loc[g["prec"] == "CsI", "amt"] = 0.07
    A, X = {}, {}
    for _, r in g.iterrows():
        mol = r["amt"] * MOLAR[str(r["unit"]).strip().lower()]
        spec = ION[r["prec"]]
        for i, k in spec.get("A", {}).items(): A[i] = A.get(i, 0) + mol * k
        for i, k in spec.get("X", {}).items(): X[i] = X.get(i, 0) + mol * k
    sA, sX = sum(A.values()), sum(X.values())
    A = {k: round(v / sA, 3) for k, v in A.items()}
    X = {k: round(v / sX, 3) for k, v in X.items()}
    _pm = psk_composition["device_id"] == dev
    for ion, val in A.items(): psk_composition.loc[_pm, f"a_{ion}"] = val
    for ion, val in X.items(): psk_composition.loc[_pm, f"x_{ion}"] = val
    psk_composition.loc[_pm, "precursor_status"] = "ok (molar) | §8.9: CsI 0.7->0.07 (typo ในเปเปอร์)"
    comp_dev.loc[dev, ["a_Cs", "a_FA", "x_Br"]] = [A.get("Cs", 0), A.get("FA", 0), X.get("Br", 0)]
    print(f"  {dev}: a_Cs={A.get('Cs')} a_FA={A.get('FA')} a_MA={A.get('MA')} x_Br={X.get('Br')}")

assert abs(comp_dev.loc["10_1002_solr_202200422_Sample1", "a_Cs"] - 0.048) < 0.005, "a_Cs ไม่ใช่ ~0.048"
print("gate ผ่าน: composition ใหม่ = สูตรมาตรฐาน Cs05FA81MA14")

แก้ CsI 0.7 -> 0.07 M: 2 แถว (Sample1, Sample2)
  10_1002_solr_202200422_Sample1: a_Cs=0.048 a_FA=0.81 a_MA=0.143 x_Br=0.14
  10_1002_solr_202200422_Sample2: a_Cs=0.048 a_FA=0.81 a_MA=0.143 x_Br=0.14
gate ผ่าน: composition ใหม่ = สูตรมาตรฐาน Cs05FA81MA14


## 9. สรุปสิ่งที่ควรใช้

| ฝั่ง | ใช้ | จำนวน | หมายเหตุ |
|---|---|---|---|
| **Y ประสิทธิภาพ** | `voc` `jsc` `ff` แยก คูณเป็น pce | 43 dev / 15 paper | variant-consistent (`jv_variant` กำกับ champion/plain/average) |
| **mediator** | `vtfl` | ~28 dev / 11 paper | แทน grain_size (13) |
| **Y เสถียร** | `retention_25c` | 14 dev | exploratory เท่านั้น |
| **X knob** | `psk_spin_rpm` `psk_anneal_C` `psk_anneal_time_s` `antisolvent_vol_uL` **`antisolvent_drop_frac`** `htl_anneal_C` | 30-43 dev | §4-6.1 merge เข้า target_table แล้ว |
| **X ควบคุม** | `a_Cs` `a_FA` `x_Br` + `etl_family` | 44 dev | §8.5-8.6 กัน confound สูตรเคมี + สถาปัตยกรรม ETL |

**ตัดทิ้ง**: PCE ตรง (= ผลคูณ), grain_size แกนหลัก, t80, has_antisolvent/etl (paper เดียว), *_family variance=0 (ยกเว้น `etl_family` L6 — PCBM/C60 มี variance ข้าม paper เก็บใน §8.6)

**ระวัง**: ใช้ `antisolvent_drop_frac` (รวมสเกลแล้ว) ไม่ใช่ `antisolvent_timing_s` ดิบที่มี 2 จุดอ้างอิง — และอย่าใส่ทั้งคู่พร้อมกัน เป็นข้อมูลเดียวกัน

**โมเดล**: RandomForest + SHAP, LeaveOneGroupOut(paper) 15 fold, รายงาน mean±SD + bootstrap, ผล = hypothesis

In [20]:
# ---- X: knob ระดับ device (PSK layer 4 + HTL layer 2) ----
# filter source_sheet: "NiOx synthesis" = สูตรทำผง NiOx ก่อนละลายเป็นน้ำหมึก ไม่ใช่ขั้นตอนสร้างเซลล์
# มีแค่ DOI เดียว (10.1002admi.202500231, 4 dev) -> confound 100% กับ paper นั้น ห้ามเป็น feature
_cell = pr_out[pr_out["source_sheet"] == "Process"]
_lidn = pd.to_numeric(_cell["layer_id"], errors="coerce")
_pskp = _cell[_lidn == 4]
_htlp = _cell[_lidn == 2]

def _knob(df, ptype, col, agg):
    if col not in df.columns:
        return pd.Series(dtype=float)
    g = df[df["process_type"] == ptype].groupby("device_id")[col]
    # min_count=1: กลุ่มที่เป็น NaN ล้วนต้องได้ NaN ไม่ใช่ 0
    # (pandas .sum() คืน 0.0 ให้กลุ่ม all-NaN -> อ่านเป็น "อบ 0 วินาที" ทั้งที่แปลว่า "ไม่รู้")
    return g.sum(min_count=1) if agg == "sum" else getattr(g, agg)()

X_dev = pd.DataFrame(index=device_layer_flags.set_index("device_id").index)
# audit ข้อ N: เดิมใช้ "max" โดยสมมติว่า step สุดท้าย = RPM สูงสุด ซึ่งเป็นการเดา ไม่ใช่การอ่านลำดับ
#   และคนละกติกากับ psk_spin_last_s ที่เรียงตาม step_order จริง -> ทำให้สองคอลัมน์อาจมาจากคนละขั้น
#   ตอนนี้ใช้ขั้นสุดท้ายจริงทั้งคู่ + assert ว่าสมมติฐานเดิมยังเป็นจริง (ถ้าวันหนึ่งไม่จริงจะได้รู้)
_sp_rows = _pskp[(_pskp["process_type"] == "Spin coating") & _pskp["Spin coating__RPM"].notna()]
_rpm_last = (_sp_rows.sort_values("step_order").groupby("device_id").tail(1)
             .set_index("device_id")["Spin coating__RPM"])
_rpm_max  = _sp_rows.groupby("device_id")["Spin coating__RPM"].max()
_mismatch = _rpm_last[(_rpm_last - _rpm_max.reindex(_rpm_last.index)).abs() > 1e-9]
if len(_mismatch):
    print(f"หมายเหตุ: {len(_mismatch)} device ที่ขั้นสุดท้ายไม่ใช่ RPM สูงสุด -> ใช้ค่าขั้นสุดท้าย")
    display(pd.DataFrame({"last": _rpm_last[_mismatch.index], "max": _rpm_max[_mismatch.index]}))
X_dev["psk_spin_rpm"]         = _rpm_last
X_dev["psk_anneal_C"]         = _knob(_pskp, "Annealing", "Annealing__C", "max")
X_dev["psk_anneal_time_s"]    = _knob(_pskp, "Annealing", "Annealing__time_s", "sum")
X_dev["antisolvent_vol_uL"]   = _knob(_pskp, "AntiSolvent", "AntiSolvent__vol_uL", "max")
X_dev["antisolvent_timing_s"] = _knob(_pskp, "AntiSolvent", "AntiSolvent__timing_s", "first")
X_dev["antisolvent_timing_ref"] = (_pskp.dropna(subset=["AntiSolvent__timing_s"])
                                   .groupby("device_id")["AntiSolvent__timing_ref"].first())
X_dev["htl_anneal_C"]         = _knob(_htlp, "Annealing", "Annealing__C", "max")

# ---- รวมสเกล antisolvent timing ให้เป็นแกนเดียว ----
# ดิบมี 2 จุดอ้างอิง: บวก = วินาทีหลังเริ่ม spin ขั้นสุดท้าย / ลบ = วินาทีก่อนจบ
# แปลงตัวลบด้วยความยาว spin ขั้นสุดท้าย -> ได้ "วินาทีนับจากเริ่ม spin ขั้นสุดท้าย" เหมือนกันหมด
_spin = (_pskp[(_pskp["process_type"] == "Spin coating") & _pskp["Spin coating__sec"].notna()]
         .sort_values("step_order").groupby("device_id").tail(1)
         .set_index("device_id")["Spin coating__sec"])
X_dev["psk_spin_last_s"] = _spin

_t = X_dev["antisolvent_timing_s"]
X_dev["antisolvent_drop_s"] = _t.where(_t >= 0, _spin.reindex(X_dev.index) + _t)
X_dev["antisolvent_drop_frac"] = X_dev["antisolvent_drop_s"] / X_dev["psk_spin_last_s"]

# audit ข้อ L: คำว่า "หลังเริ่มปั่น" ของค่าบวกมีจุดอ้างอิงได้ 2 จุด และโค้ดเดิมเลือกให้เองโดยไม่เคยตรวจกับเปเปอร์
#   (จุดอ้างอิงถูกกำหนดจากเครื่องหมายของตัวเลขล้วน ๆ ใน §6.1 -- เป็นกฎที่เราตั้ง ไม่ใช่ข้อเท็จจริงจากต้นทาง)
#   ค่าลบไม่กำกวมเพราะยึดจุดจบ ซึ่งเป็นจุดเดียวกันทั้งสองแบบ -- ปัญหาอยู่ที่ค่าบวกเท่านั้น
#   แทนที่จะเดา ให้เก็บทั้งสองการตีความไว้ แล้วให้ขั้น modeling ทดสอบความไวเอาเองว่าผลเปลี่ยนไหม
#     antisolvent_drop_frac      = นับจากเริ่ม spin ขั้นสุดท้าย   (การตีความปัจจุบัน)
#     antisolvent_drop_frac_alt  = นับจากเริ่มโปรแกรม spin ทั้งหมด (อีกการตีความ)
_spin_total = (_pskp[(_pskp["process_type"] == "Spin coating") & _pskp["Spin coating__sec"].notna()]
               .groupby("device_id")["Spin coating__sec"].sum(min_count=1))
X_dev["psk_spin_total_s"] = _spin_total
X_dev["antisolvent_drop_frac_alt"] = X_dev["antisolvent_drop_s"] / X_dev["psk_spin_total_s"]

_cmp = X_dev.dropna(subset=["antisolvent_drop_frac", "antisolvent_drop_frac_alt"])
_gap = (_cmp["antisolvent_drop_frac"] - _cmp["antisolvent_drop_frac_alt"]).abs()
print(f"จังหวะหยด 2 การตีความ: ต่างกันสูงสุด {_gap.max():.3f} | "
      f"{int((_gap > 0.01).sum())}/{len(_cmp)} device ที่ต่างกันเกิน 0.01")
if (X_dev["antisolvent_drop_frac"] >= 0.999).any():
    _edge = X_dev.index[X_dev["antisolvent_drop_frac"] >= 0.999]
    print(f"  ⚠ {list(_edge)} ได้สัดส่วน 1.00 พอดี = หยดวินาทีสุดท้ายของการปั่น ซึ่งขัดกับกลไก")
    print(f"    อีกการตีความให้ {X_dev.loc[_edge, 'antisolvent_drop_frac_alt'].round(3).to_dict()} ซึ่งสมเหตุผลกว่า")
    print("    -> ต้องเปิดเปเปอร์ชี้ขาด ระหว่างนี้ใช้ทั้งสองคอลัมน์ทดสอบความไว")

# gate: จุดหยดต้องอยู่ในช่วง spin จริง (0 < t <= ความยาว spin)
_chk = X_dev.dropna(subset=["antisolvent_drop_s", "psk_spin_last_s"])
assert ((_chk["antisolvent_drop_s"] > 0) & (_chk["antisolvent_drop_s"] <= _chk["psk_spin_last_s"])).all(),     "จุดหยด antisolvent หลุดนอกช่วง spin"

_n_conv = int((_t < 0).sum())
print(f"antisolvent_drop_s: {int(X_dev['antisolvent_drop_s'].notna().sum())} dev "
      f"(แปลงจาก before_end {_n_conv} dev) | frac: {int(X_dev['antisolvent_drop_frac'].notna().sum())} dev")
display(X_dev.loc[_t.notna(), ["psk_spin_last_s", "antisolvent_timing_s",
                               "antisolvent_timing_ref", "antisolvent_drop_s",
                               "antisolvent_drop_frac"]].round(3).head(12))

# target_table = flags + Y + stability + X knob + composition + สถาปัตยกรรม (etl_family §8.6)
target_table = (device_layer_flags.set_index("device_id")
                .join(Y).join(stab_Y).join(X_dev).join(comp_dev).join(arch_dev))

assert len(target_table) == 44 and target_table.index.is_unique, "target_table ต้อง 44 แถว unique"
print("target_table:", target_table.shape)
print("\ncoverage ต่อคอลัมน์ (devices):")
display(target_table.notna().sum().rename("n_devices").to_frame().T)
display(target_table.head())
# ยังไม่เขียน -- รวมเขียนทีเดียวใน §10

จังหวะหยด 2 การตีความ: ต่างกันสูงสุด 0.250 | 26/31 device ที่ต่างกันเกิน 0.01
  ⚠ ['10_1021_acsami_5c15868_Sample1', '10_1021_acsami_5c15868_Sample2', '10_1021_acsami_5c15868_Sample3'] ได้สัดส่วน 1.00 พอดี = หยดวินาทีสุดท้ายของการปั่น ซึ่งขัดกับกลไก
    อีกการตีความให้ {'10_1021_acsami_5c15868_Sample1': 0.75, '10_1021_acsami_5c15868_Sample2': 0.75, '10_1021_acsami_5c15868_Sample3': 0.75} ซึ่งสมเหตุผลกว่า
    -> ต้องเปิดเปเปอร์ชี้ขาด ระหว่างนี้ใช้ทั้งสองคอลัมน์ทดสอบความไว
antisolvent_drop_s: 33 dev (แปลงจาก before_end 6 dev) | frac: 31 dev


,psk_spin_last_s,antisolvent_timing_s,antisolvent_timing_ref,antisolvent_drop_s,antisolvent_drop_frac
device_id,,,,,
10.1002admi.202500231_Sample1,35.0,-10.0,before_end,25.0,0.714
10.1002admi.202500231_Sample2,35.0,-10.0,before_end,25.0,0.714
10.1002admi.202500231_Sample3,35.0,-10.0,before_end,25.0,0.714
10.1002admi.202500231_Sample4,35.0,-10.0,before_end,25.0,0.714
10.1016j.jechem.2021.11.029_Sample1,40.0,8.0,after_start,8.0,0.200
10.1016j.jechem.2021.11.029_Sample2,40.0,8.0,after_start,8.0,0.200
10.1021_acsami.5c16972_Sample2,40.0,-5.0,before_end,35.0,0.875
10.1021_acsami.5c16972_Sample3,40.0,-5.0,before_end,35.0,0.875
10_1002_smll_202504237_Sample1,30.0,15.0,after_start,15.0,0.500


target_table: (44, 42)

coverage ต่อคอลัมน์ (devices):


,is_coated,n_coating_layers,coat_htl,coat_psk,coat_etl,anneal_htl,anneal_psk,htl_mod_kind,psk_mod_kind,jv_variant,jv_scan,pce,voc,jsc,ff,rrec,rs,vtfl,trap_density,trap_density_electron,defect_density_reported,trap_implied_thickness_nm,trap_scale_flag,rrec_scale_flag,retention_25c,degradation_rate,psk_spin_rpm,psk_anneal_C,psk_anneal_time_s,antisolvent_vol_uL,antisolvent_timing_s,antisolvent_timing_ref,htl_anneal_C,psk_spin_last_s,antisolvent_drop_s,antisolvent_drop_frac,psk_spin_total_s,antisolvent_drop_frac_alt,a_Cs,a_FA,x_Br,etl_family
n_devices,44,44,44,44,44,44,44,29,22,43,43,42,43,43,43,15,20,28,22,3,2,21,24,15,14,14,44,42,42,35,33,33,42,42,33,31,42,31,44,44,44,44


,is_coated,n_coating_layers,coat_htl,coat_psk,coat_etl,anneal_htl,anneal_psk,htl_mod_kind,psk_mod_kind,jv_variant,jv_scan,pce,voc,jsc,ff,rrec,rs,vtfl,trap_density,trap_density_electron,defect_density_reported,trap_implied_thickness_nm,trap_scale_flag,rrec_scale_flag,retention_25c,degradation_rate,psk_spin_rpm,psk_anneal_C,psk_anneal_time_s,antisolvent_vol_uL,antisolvent_timing_s,antisolvent_timing_ref,htl_anneal_C,psk_spin_last_s,antisolvent_drop_s,antisolvent_drop_frac,psk_spin_total_s,antisolvent_drop_frac_alt,a_Cs,a_FA,x_Br,etl_family
device_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
10.1002admi.202500231_Sample1,True,1,False,False,True,True,True,NaN,NaN,plain,-,16.31,1.020,20.97,76.00,NaN,0.68,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4000.0,100.0,1800.0,250.0,-10.0,before_end,270.0,35.0,25.0,0.714286,45.0,0.555556,0.05,0.790,0.13,C60
10.1002admi.202500231_Sample2,True,1,False,False,True,True,True,additive,NaN,plain,-,17.11,1.040,22.01,75.00,NaN,0.64,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4000.0,100.0,1800.0,250.0,-10.0,before_end,270.0,35.0,25.0,0.714286,45.0,0.555556,0.05,0.790,0.13,C60
10.1002admi.202500231_Sample3,True,1,False,False,True,True,True,additive,NaN,plain,-,16.97,1.000,21.86,78.00,NaN,0.62,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4000.0,100.0,1800.0,250.0,-10.0,before_end,270.0,35.0,25.0,0.714286,45.0,0.555556,0.05,0.790,0.13,C60
10.1002admi.202500231_Sample4,True,1,False,False,True,True,True,additive,NaN,plain,-,16.25,0.990,20.79,79.00,NaN,0.53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4000.0,100.0,1800.0,250.0,-10.0,before_end,270.0,35.0,25.0,0.714286,45.0,0.555556,0.05,0.790,0.13,C60
10.1016j.jechem.2021.11.029_Sample1,True,2,False,True,True,True,True,NaN,coating,plain,RS,17.83,1.064,21.32,78.58,NaN,NaN,0.66,6.080000e+15,NaN,NaN,619.653159,ok,NaN,NaN,NaN,5000.0,100.0,1800.0,200.0,8.0,after_start,100.0,40.0,8.0,0.200000,50.0,0.160000,0.05,0.807,0.15,PCBM


## 9.5 ด่านตรวจรวมศูนย์ก่อน export — ย้ายมาจาก §8.7 (audit ข้อ P, Q, U)

In [21]:
# ---- §9.5 ด่านตรวจรวมศูนย์ ก่อน export (audit ข้อ P, Q, U) ----
# ข้อ Q: เดิมด่านตรวจ value_parsed อยู่ที่ §8.7 แล้วมี §8.8/§8.9 แก้ค่าตามมาทีหลังโดยไม่ตรวจซ้ำ
#        -> ย้ายมาไว้จุดสุดท้ายจริง ที่ทุกการแก้เกิดขึ้นครบแล้ว
# ข้อ P: เดิมเงื่อนไขบังคับว่าต้องมีการแก้หน่วยมาก่อน ซึ่ง "ยกเว้น" แถว FF พอดี แล้วประกาศว่าตรวจครบ
#        -> ตรวจทุกแถวที่สองคอลัมน์ไม่ตรงกัน แล้วบอกให้ชัดว่าอันไหนตั้งใจ อันไหนไม่
# ข้อ U: ตัวตรวจหน่วยเดิมหาแค่ชื่อ voc/jsc/mobility -> เปลี่ยนเป็นตรวจตามช่วงค่าที่เป็นไปได้ทางฟิสิกส์
_pp = tp["properties"]
_v  = pd.to_numeric(_pp["value"], errors="coerce")
_vc = pd.to_numeric(_pp["value_after_unit_fix"], errors="coerce")
_vp = pd.to_numeric(_pp["value_parsed"], errors="coerce")

# 1) value_parsed ต้องอธิบายได้เสมอว่าทำไมต่างจาก value_after_unit_fix
_diff = _vc.notna() & _vp.notna() & ~np.isclose(_vp, _vc, rtol=1e-9)
_explained = _pp["value_note"].fillna("").str.strip() != ""
_unexplained = _diff & ~_explained
print(f"value_parsed ต่างจาก value_after_unit_fix: {int(_diff.sum())} แถว "
      f"(มีบันทึกเหตุผลครบ {int((_diff & _explained).sum())} แถว)")
assert not _unexplained.any(), \
    "มีแถวที่ value_parsed ต่างจาก value_after_unit_fix โดยไม่มี value_note อธิบาย"

# 2) ช่วงค่าที่เป็นไปได้ทางฟิสิกส์ -- ดักได้ทุก property โดยไม่ต้องรู้ชื่อล่วงหน้า
PHYS_RANGE = {   # base_property: (ต่ำสุด, สูงสุด, เหตุผล)
    "Voc":          (0.5, 2.0,   "แรงดันเปิดวงจรของเซลล์ perovskite"),
    "Jsc":          (5.0, 30.0,  "กระแสลัดวงจร จำกัดด้วย bandgap"),
    "FF":           (30.0, 90.0, "fill factor เป็น % ไม่ใช่เศษส่วน"),
    "PCE":          (1.0, 30.0,  "ประสิทธิภาพรวม"),
    "VTFL":         (0.01, 5.0,  "แรงดัน trap-filled limit"),
    "Trap density": (1e13, 1e19, "ความหนาแน่นกับดักในฟิล์ม"),
    "Work Function":(3.0, 6.5,   "work function อ้างอิงสุญญากาศ (บวก)"),
}
_bad_rows = []
for bp, (lo, hi, why) in PHYS_RANGE.items():
    m = (_pp["base_property"] == bp) & _vp.notna() & ((_vp < lo) | (_vp > hi))
    for i in _pp.index[m]:
        _bad_rows.append({"device_id": _pp.at[i, "device_id"], "property": _pp.at[i, "property"],
                          "value_parsed": _vp[i], "ช่วงที่ควรเป็น": f"{lo:g} ถึง {hi:g}", "เหตุผล": why})

# ระดับพลังงานที่อ้างอิงสุญญากาศต้องเป็นค่าลบเสมอ (audit ข้อ C จับได้ที่นี่)
_vac = _pp["base_property"].isin(["VBM", "Fermi level", "VB"]) & _vp.notna() & (_vp > 0)
for i in _pp.index[_vac]:
    _bad_rows.append({"device_id": _pp.at[i, "device_id"], "property": _pp.at[i, "property"],
                      "value_parsed": _vp[i], "ช่วงที่ควรเป็น": "< 0",
                      "เหตุผล": "ระดับพลังงานอ้างอิงสุญญากาศต้องเป็นลบ"})
if _bad_rows:
    display(pd.DataFrame(_bad_rows))
assert not _bad_rows, "มีค่าที่หลุดช่วงเป็นไปได้ทางฟิสิกส์ -- ต้องตรวจกับต้นฉบับก่อนไปต่อ"
print("ด่านตรวจช่วงค่าทางฟิสิกส์: ผ่านทุกแถว")

# 3) knob ที่กู้จาก description ต้องเข้ามาครบตามที่ตั้งใจ
_rec = target_table.loc["10_1021_acsami_5c15868_Sample1"]
assert _rec["psk_spin_rpm"] == 4000 and _rec["psk_anneal_C"] == 100 and _rec["htl_anneal_C"] == 200, \
    "ค่าที่กู้จาก description ของ acsami_5c15868_Sample1 ไม่เข้า target_table"
assert target_table.loc["10_1002_solr_202400534_Sample2", "psk_spin_rpm"] == 5000, \
    "psk_spin_rpm ของ solr_202400534_Sample2 ต้องเป็น 5000 (audit ข้อ A)"
print("ด่านตรวจค่าที่กู้คืน: ผ่าน")

value_parsed ต่างจาก value_after_unit_fix: 11 แถว (มีบันทึกเหตุผลครบ 11 แถว)
ด่านตรวจช่วงค่าทางฟิสิกส์: ผ่านทุกแถว
ด่านตรวจค่าที่กู้คืน: ผ่าน


## 10. Export `prepared_v2` — ชุดเต็มทุกไฟล์

`prepared_v2/` = ทุกตารางจาก `prepared/` + สิ่งที่ recheck เพิ่ม

| ไฟล์ | เปลี่ยนจาก `prepared/` |
|---|---|
| `layers.csv` | + `layer_kind` (map ตรงจาก layer_id — **ไม่ merge** กันแถวพองเคส layer 7 สองแถว PCBM+BCP) |
| `process.csv` | + knob columns ที่แตก + `param_fixed` (§4.0) + `param_note` + `AntiSolvent__vol_uL/timing_s/timing_ref` (§6.1) + `*__time_s` (§6) |
| `devices.csv` | + layer flags (is_coated, coat_*, anneal_*) |
| `materials/properties/stability/vendors.csv` | เหมือนเดิม |
| `target_table.csv` | ใหม่ — Y (variant-consistent) + mediator + stability + **X knob + composition** |
| `device_layer_flags.csv` | ใหม่ |
| `psk_composition.csv` | ใหม่ — §8.5 (เดิมคำนวณใน column_prepare แต่ไม่เคย export) |

In [22]:
V2 = ROOT / "processed" / "prepared_v2"
V2.mkdir(parents=True, exist_ok=True)

# 1) layers + layer_kind -- map ตรงจาก layer_id ห้าม merge
#    (solr_202400534 มี layer 7 สองแถว PCBM+BCP -- merge บน (device_id, layer_id) จะคูณแถวเป็น 2x2)
_lay_out = lay.copy()
_kind = _lay_out["layer_id"].map(
    lambda i: "main" if i in MAIN_IDS else "coating" if i in COATING_IDS else "other")
_lay_out["layer_kind"] = _kind.where(_lay_out["material"].notna())
# แถวที่เป็น additive (ผสมในน้ำหมึก ไม่ได้เคลือบทับ) ต้องไม่ถูกเรียกว่า coating -- ดู §2.1
for _lid, _col in MOD_KIND.items():        # ใช้ device_layer_flags -- `dev` ถูก loop ใน §7/§8.5 เขียนทับ
    _adv = device_layer_flags.loc[device_layer_flags[_col] == "additive", "device_id"]
    _m = (_lay_out["layer_id"] == _lid) & _lay_out["device_id"].isin(_adv)
    _lay_out.loc[_m, "layer_kind"] = "additive"
print("layer_kind:", _lay_out["layer_kind"].value_counts(dropna=False).to_dict())
assert len(_lay_out) == len(lay), "layers แถวพอง"
_lay_out.to_csv(V2 / "layers.csv", index=False)

# 2) process = แตก parameter + PARAM_FIX + antisolvent + time_s
assert int((pr_out["param_note"] != "").sum()) == 0, "ไม่ควรเหลือแถวติดธงแล้ว (ปิดด้วย SI)"
pr_out.to_csv(V2 / "process.csv", index=False)

# 3) devices + layer/anneal flags
_dev_out = tp["devices"].merge(device_layer_flags, on="device_id", how="left")
_dev_out.to_csv(V2 / "devices.csv", index=False)

# 4) ตารางที่เหลือ copy ตรง
for name in ["materials", "properties", "stability", "vendors"]:
    tp[name].to_csv(V2 / f"{name}.csv", index=False)

# 5) ตารางที่ recheck สร้าง
target_table.to_csv(V2 / "target_table.csv")
device_layer_flags.to_csv(V2 / "device_layer_flags.csv", index=False)
psk_composition.to_csv(V2 / "psk_composition.csv", index=False)

print("wrote prepared_v2:")
for p in sorted(V2.glob("*.csv")):
    d = pd.read_csv(p)
    print(f"  {p.name:24s} {d.shape}")

# verify จากไฟล์บนดิสก์
_chk_lay = pd.read_csv(V2 / "layers.csv")
assert len(_chk_lay) == len(lay), "layers บนดิสก์แถวไม่ตรง"
_chk_tt = pd.read_csv(V2 / "target_table.csv")
assert len(_chk_tt) == 44 and _chk_tt["device_id"].is_unique, "target_table บนดิสก์ไม่ใช่ 44 unique"
_hv = _chk_tt.dropna(subset=["voc", "jsc", "ff", "pce"])
_e = (_hv["voc"] * _hv["jsc"] * _hv["ff"] / 100 - _hv["pce"]).abs()
# audit ข้อ O/V: หลังคุมทั้ง variant และทิศทางสแกนแล้ว ส่วนต่างที่เหลือต้องอธิบายได้ทุกตัว
#   เกณฑ์เดิม 1.0 หลวมเกินไปจนซ่อนการปนทิศทางสแกนไว้ได้ -> ลดเป็น 0.2 (ระดับที่การปัดเศษทำได้จริง)
#   2 เครื่องที่ยังเกินมีเหตุผลรองรับ ไม่ใช่การปนชุดค่า:
#     ccschem_Sample1 0.503 -- ตัวเลขในเปเปอร์เองไม่สอดคล้อง (1.15 x 23.53 x 76.40 = 20.67 แต่รายงาน 20.17)
#     heliyon_Sample3 0.302 -- ค่ารายงานปัดหยาบ (1.06 / 25.00 / 78.5) รวมกันแล้วคลาดได้ถึงระดับนี้
_ERR_KNOWN = {"10_31635_ccschem_025_202505977_Sample1", "10_1016_j_heliyon_2024_e25352_Sample3"}
_e = _e.set_axis(_hv["device_id"].values)          # index เป็น device_id เพื่ออ้างชื่อเครื่องได้
_e_over = _e[_e > 0.2]
_e_new = set(_e_over.index) - _ERR_KNOWN
assert not _e_new, f"มีเครื่องที่ pce ไม่ตรงสมการเกิน 0.2 และยังไม่มีคำอธิบาย: {sorted(_e_new)}"
print(f"gate pce=voc*jsc*ff/100: max_err {_e.max():.3f} | เกิน 0.2: {len(_e_over)} เครื่อง (อธิบายได้ทั้งหมด)")
# gate: knob ใน target_table ต้องไม่มีค่าที่มาจาก NiOx synthesis (confound 100% กับ DOI เดียว)
_ns_dev = set(pr_out.loc[pr_out["source_sheet"] == "NiOx synthesis", "device_id"])
_KNOB = ["psk_spin_rpm", "psk_anneal_C", "psk_anneal_time_s",
         "antisolvent_vol_uL", "antisolvent_timing_s", "htl_anneal_C"]
_ref = (_chk_tt.set_index("device_id")[_KNOB]
        .compare(target_table[_KNOB]) if False else None)   # (เทียบตรงผ่าน assert ด้านล่างแทน)
_cell_only = pr_out[pr_out["source_sheet"] == "Process"]
_lidn2 = pd.to_numeric(_cell_only["layer_id"], errors="coerce")
_htl_ref = (_cell_only[(_lidn2 == 2) & (_cell_only["process_type"] == "Annealing")]
            .groupby("device_id")["Annealing__C"].max())
_htl_now = _chk_tt.set_index("device_id")["htl_anneal_C"].dropna()
assert _htl_now.round(6).equals(_htl_ref.reindex(_htl_now.index).round(6)),     "htl_anneal_C ไม่ตรงกับค่าที่คำนวณจาก source_sheet='Process' อย่างเดียว"
print(f"gate NiOx synthesis ผ่าน: knob ทั้ง {len(_KNOB)} ตัวมาจาก source_sheet='Process' เท่านั้น "
      f"({len(_ns_dev)} device ที่มีชีตสังเคราะห์ไม่ปนเข้ามา)")

print(f"verify ผ่าน: layers {len(_chk_lay)} แถวไม่พอง | target_table 44 unique | pce consistency max_err={_e.max():.3f}")

layer_kind: {'main': 218, 'coating': 94, nan: 37, 'additive': 3}


wrote prepared_v2:
  device_layer_flags.csv   (44, 10)
  devices.csv              (44, 13)
  layers.csv               (352, 11)
  materials.csv            (543, 13)
  process.csv              (702, 72)
  properties.csv           (563, 16)
  psk_composition.csv      (44, 11)
  stability.csv            (62, 20)
  target_table.csv         (44, 43)
  vendors.csv              (644, 8)


gate pce=voc*jsc*ff/100: max_err 0.503 | เกิน 0.2: 2 เครื่อง (อธิบายได้ทั้งหมด)


gate NiOx synthesis ผ่าน: knob ทั้ง 6 ตัวมาจาก source_sheet='Process' เท่านั้น (4 device ที่มีชีตสังเคราะห์ไม่ปนเข้ามา)
verify ผ่าน: layers 352 แถวไม่พอง | target_table 44 unique | pce consistency max_err=0.503
